# RAG Evaluation: Measuring What Your Pipeline Actually Gets Wrong

## The Brief: Proving the Metrics Before They Guard Riverside's Catalog

Riverside House's knowledge-base assistant is almost ready to answer questions over confidential manuscripts. Before release, the engineering team needs to know whether it retrieved the wrong source, ignored good context, drifted off topic, or produced an incorrect answer. Fluent text alone cannot distinguish those failures.

Rather than trust an untested harness against private manuscripts, Riverside rehearses on a safe stand-in corpus: **14 internal documents about LLM techniques and infrastructure**, with **8 labeled question/reference/gold-document cases**. If a metric cannot expose known failures here, it should not gate the manuscript catalog.

This notebook builds the RAG evaluation mental model from failure to measurement. Every metric is exercised on the same running question:

> **How does ReAct combine reasoning and acting?**

The CPU profile uses MiniLM embeddings and the CUDA profile uses MPNet. Exact cosine values and gaps can therefore differ by hardware; the code prints measured values and branches its interpretation instead of promising one fixed number.

| Step | Concept | Riverside's question | Evidence this notebook seeks |
| --- | --- | --- | --- |
| 1 | Four RAG failure modes | Can fluent text reveal where the system failed? | Construct failures that look plausible but have different causes |
| 2 | Context recall, precision, and relevance | Did the assistant retrieve the right document? | Compare labels and semantic relevance; expose keyword-overlap limits |
| 3 | Groundedness | Did the answer use the retrieved facts? | Detect invented vocabulary and demonstrate the negation/entailment ceiling |
| 4 | Answer relevance | Did the answer address the question? | Measure whether on-topic and off-topic answers separate under the selected encoder |
| 5 | Correctness and ROUGE-L | Does paraphrasing receive fair credit? | Compare Jaccard and ordered subsequence coverage, then expose synonym limits |
| 6 | Composite dashboard | Which component is the bottleneck? | Show different metric fingerprints rather than trusting one average |
| 7 | LLM-as-judge | What replaces proxies for product decisions? | Preserve the evaluator structure while upgrading the scorer and auditing judge bias |

A small fixture can prove metric mechanics and failure boundaries. It cannot calibrate production thresholds; those require representative Riverside labels, scorer versions, and repeated evaluation.

---

## Locate the Failure Before Scoring It

![RAG failure location map from question through retrieval, context, generation, and answer](images/rag-failure-location-map.png)

Evaluate the stage that can explain the observed failure: retrieval coverage and ranking, context relevance and grounding, then answer correctness. Metrics such as Recall@k are diagnostic measurements, not interchangeable with a single hit-rate definition.


## Table of Contents

1. [The Brief: Proving the Metrics Before They Guard Riverside's Catalog](#the-brief-proving-the-metrics-before-they-guard-riversides-catalog)
   - [The Full Landscape of RAG Evaluation (and What This Notebook Covers)](#the-full-landscape-of-rag-evaluation-and-what-this-notebook-covers)
2. [Part 1 - The Four Silent Failures of RAG](#part-1--the-four-silent-failures-of-rag)
3. [Part 2 - Retrieval Quality: Context Recall and Context Precision](#part-2--retrieval-quality-context-recall-and-context-precision)
4. [Part 3 - Groundedness: Does Every Claim Trace Back to the Context?](#part-3--groundedness-does-every-claim-trace-back-to-the-context)
5. [Part 4 - Answer Relevance: Does the Answer Address the Question?](#part-4--answer-relevance-does-the-answer-address-the-question)
6. [Part 5 - Correctness: ROUGE-L and the Ordering Problem](#part-5--correctness-rouge-l-and-the-ordering-problem)
7. [Part 6 - Composite Dashboard: Diagnosing the Bottleneck](#part-6--composite-dashboard-diagnosing-the-bottleneck)
8. [Part 6b - Oracle Context: Hand the Generator the Right Page](#part-6b--oracle-context-hand-the-generator-the-right-page)
9. [Part 7 - From Proxies to Production: LLM-as-Judge](#part-7--from-proxies-to-production-llm-as-judge)
   - [Judge Biases: Why "Just Ask an LLM" Isn't Automatically Trustworthy](#judge-biases-why-just-ask-an-llm-isnt-automatically-trustworthy)
   - [Two Release Questions the Average Cannot Answer](#two-release-questions-the-average-cannot-answer)
10. [Summary - The Complete RAG Evaluation Journey](#summary--the-complete-rag-evaluation-journey)
11. [What This Notebook Covered (and What It Didn't)](#what-this-notebook-covered-and-what-it-didnt)
12. [The Decision: What Does Riverside Actually Turn On in Production?](#the-decision-what-does-riverside-actually-turn-on-in-production)

> Links jump to the matching heading below. If a link does not scroll correctly in your Jupyter viewer, use `Ctrl+F` or the notebook outline with the section title.

## The Full Landscape of RAG Evaluation (and What This Notebook Covers)

**Riverside's question before we build anything:** are we picking these four metrics because
they're the right ones, or just because they're the ones that happened to come to mind first?

"RAG evaluation" is bigger than the four proxy metrics this notebook builds. Before writing a
single line of code, here is the full set of techniques a genuinely complete treatment of the
subject would at least address, grouped by the question each one answers:

| Category                                                                                   | What it answers                                                | This notebook                                                                                                                                                                                                                                                                                                          |
| ------------------------------------------------------------------------------------------ | -------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Retrieval-quality metrics** — Recall@k, Precision@k, MRR, nDCG, context recall/precision | Did we retrieve the right document(s)?                         | [Built] embedding-cosine proxy + binary recall@1 built and stress-tested; [Partial] the general (multi-doc, varying-`k`) recall/precision formulas are stated but only demoed in the degenerate single-gold-doc case; [Named only] MRR and nDCG need graded/multi-relevant-document judgments this corpus doesn't have |
| **Generation faithfulness** — groundedness/hallucination detection, NLI entailment         | Did the generator actually use the retrieved text?             | [Built] token-recall groundedness proxy built, stress-tested against an adversarial synonym case and a verbatim-copy gaming attempt                                                                                                                                                                                    |
| **Answer relevance**                                                                       | Does the answer address the question asked?                    | [Built] query-answer embedding cosine, built and verified                                                                                                                                                                                                                                                              |
| **Reference-based correctness** — Jaccard, ROUGE-N/L, BLEU, BERTScore                      | Does the answer match a labeled reference?                     | [Built] Jaccard (strawman) and ROUGE-L (hand-implemented LCS) built; [Partial] ROUGE-L's own remaining ceiling (heavy vocabulary substitution) gets a short illustration further down; [Named only] BLEU and BERTScore named, not built                                                                                |
| **LLM-as-judge evaluation**                                                                | What replaces these proxies once real editors are on the line? | [Partial] the structured-output judge pattern is explained and shown as real (commented-out) reference code; [Partial] judge biases (position, self-preference, verbosity) get a short illustration in Part 7                                                                                                          |
| **Human evaluation** — annotation guidelines, inter-annotator agreement                    | Where do labels/rubrics themselves come from?                  | [Named only] named, out of scope — an organizational/process topic, not a metric to run in this notebook                                                                                                                                                                                                               |
| **Evaluation granularity** — end-to-end vs. component-only (generator given gold context)  | Which pipeline stage is actually being measured?               | [Built] every metric here diagnoses _which_ stage failed on real end-to-end output; [Named only] a true "hold retrieval fixed at the gold document" isolation ablation isn't built                                                                                                                                     |
| **Golden dataset construction**                                                            | Where do labeled examples come from?                           | [Built] 8 hand-labeled question/reference/gold-document triples built directly from this notebook's own corpus; [Named only] LLM-assisted synthetic QA generation named, not built                                                                                                                                     |
| **Regression testing / cost & latency at scale**                                           | What do we actually run in production, and how often?          | [Built] a real threshold-based release scorecard runs against this run's numbers; [Partial] the cost/latency reasoning for proxies-vs-judge is explained in the closing Decision section, not measured; [Named only] tracking metrics across multiple pipeline versions over time isn't demonstrated                   |

**[Built]** = implemented and demonstrated with real code and a verified result · **[Partial]** =
explained, with an illustrative snippet, but not fully built · **[Named only]** = named and reasoned
about, explicitly out of scope for this notebook. The complete item-by-item ledger — every technique
above, with a one-line reason for its tier — appears in **What This Notebook Covered (and What It
Didn't)** near the end.

---


In [ ]:
#  Install dependencies (run once)
import subprocess, sys

required = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("sentence_transformers", "sentence-transformers"),
    ("rank_bm25", "rank-bm25"),
]

# Only install packages that aren't already importable, to keep re-runs fast
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        print(f"  installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  ok  {pkg}")

print("\ndependencies ready")

In [ ]:
# Imports, hardware-aware embedding profile, and deterministic seeding.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import re, warnings

CUDA_AVAILABLE = torch.cuda.is_available()
MODEL_DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"
RAG_MODEL_PROFILE = "gpu-quality" if CUDA_AVAILABLE else "cpu-small"
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-mpnet-base-v2"
    if CUDA_AVAILABLE
    else "sentence-transformers/all-MiniLM-L6-v2"
)

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")
np.random.seed(42)

print(f"RAG evaluation profile: {RAG_MODEL_PROFILE} on {MODEL_DEVICE}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
if not CUDA_AVAILABLE:
    print(
        "CPU disclaimer: MiniLM keeps every metric runnable, but embedding proxies can miss "
        "negation, entailment, and subtle paraphrases. Those are expected model limitations; "
        "the notebook exposes them and does not treat proxy scores as human judgment."
    )

In [ ]:
#  Knowledge base and labeled evaluation set
DOCS = [

    #  Core LLM techniques
    "Agents are LLM-powered systems that use tools, memory, and planning to complete multi-step tasks autonomously.",
    "ReAct (Reasoning + Acting) is an agent framework that interleaves thought and action steps, using tools such as Wikipedia search or a calculator.",
    "Prompt engineering is the practice of crafting input text to guide LLM behavior, including few-shot examples, chain-of-thought, and role prompts.",
    "Few-shot prompting provides the model with 2-5 input-output examples before the target question, steering output format and reasoning style.",
    "Chain-of-thought prompting encourages step-by-step reasoning by adding worked examples, improving performance on multi-step arithmetic and logic.",
    "Adversarial attacks on LLMs include jailbreaking (bypassing safety filters), prompt injection (hijacking the system prompt), and token manipulation.",
    "Fine-tuning adapts a pretrained model to a specific task by continuing training on a curated dataset, updating model weights to shift behavior.",
    "LoRA (Low-Rank Adaptation) adds small trainable low-rank matrices to frozen pretrained weights, reducing tunable parameters by 10-100x.",

    #  Newer content: fine-tuning pipeline stages
    "Continued pretraining (non-instructional fine-tuning) trains a base model on raw domain text using next-token prediction, teaching it domain vocabulary and facts before any instruction format is introduced.",
    "DPO (Direct Preference Optimization) trains a model to prefer chosen responses over rejected ones using a closed-form loss, skipping the separate reward model required by RLHF.",

    #  Newer content: retrieval and search
    "Hybrid search combines dense semantic retrieval with sparse lexical retrieval (BM25) using Reciprocal Rank Fusion or score blending, covering both paraphrase queries and exact-term queries.",
    "BM25 is a probabilistic term-frequency ranking function that weights rare terms highly via IDF and normalises for document length; it excels at exact keyword lookup but misses synonyms.",

    #  Newer content: LLM infrastructure
    "An LLM gateway sits in front of multiple providers and handles unified routing, rate limiting, fallback on provider failure, and cost-aware traffic shaping from a single call site.",
    "Semantic caching stores the embedding of a request and returns a cached answer when a new query is sufficiently similar, reducing API cost for repeated or near-duplicate questions.",
]

# The gold standard: 8 question-answer pairs covering the full expanded knowledge base.
# REFERENCE answers define what a correct response must cover.
TEST_CASES = [
    {
        "question": "How does ReAct combine reasoning and acting?",
        "reference": "ReAct interleaves reasoning steps with actions such as Wikipedia search, letting the model observe tool outputs and refine its reasoning.",
        "gold_doc_idx": 1,  # DOCS[1] is the canonical source document
    },
    {
        "question": "What biases can arise with few-shot prompting?",
        "reference": "Few-shot prompting can introduce majority label bias, recency bias, and common token bias.",
        "gold_doc_idx": 3,
    },
    {
        "question": "What is LoRA and how does it reduce fine-tuning cost?",
        "reference": "LoRA adds small low-rank matrices to frozen weights, drastically reducing the number of trainable parameters.",
        "gold_doc_idx": 7,
    },
    {
        "question": "What types of adversarial attacks target LLMs?",
        "reference": "Adversarial attacks on LLMs include jailbreaking, prompt injection, and token manipulation.",
        "gold_doc_idx": 5,
    },
    {
        "question": "How does chain-of-thought prompting work?",
        "reference": "Chain-of-thought prompting adds worked reasoning examples to improve performance on multi-step problems.",
        "gold_doc_idx": 4,
    },
    {
        "question": "What is DPO and how does it differ from RLHF?",
        "reference": "DPO uses a closed-form loss on preference pairs to skip the separate reward model that RLHF requires.",
        "gold_doc_idx": 9,
    },
    {
        "question": "Why combine BM25 with semantic search?",
        "reference": "Hybrid search covers both exact-term queries where BM25 excels and paraphrase queries where dense embeddings excel, using RRF to merge the two ranked lists.",
        "gold_doc_idx": 10,
    },
    {
        "question": "What does an LLM gateway do?",
        "reference": "An LLM gateway handles routing, rate limiting, fallback, and cost-aware traffic shaping across multiple providers from a single call site.",
        "gold_doc_idx": 12,
    },
]

print(f"knowledge base:    {len(DOCS)} documents")
print(f"evaluation set:    {len(TEST_CASES)} labeled question-answer pairs")
print(f"\nrunning question throughout the notebook:")
print(f"  Q: {TEST_CASES[0]['question']}")
print(f"  reference: {TEST_CASES[0]['reference']}")

### Hardware-Aware Evaluation Encoder

The executable RAG bot is intentionally extractive, so this notebook does not add a local causal LLM merely to satisfy a model branch. That would introduce generation variability and distract from metric behavior.

Instead, the scorer that actually powers retrieval relevance and answer relevance follows the device:

- CPU: `sentence-transformers/all-MiniLM-L6-v2` for a fast, small baseline.
- CUDA: `sentence-transformers/all-mpnet-base-v2` for stronger semantic representations.

MiniLM can miss negation, factual distortion, or paraphrases with little vocabulary overlap. MPNet can improve semantic ranking but is still not an entailment judge. These limitations are expected and are demonstrated explicitly; the optional production section keeps a stronger hosted LLM judge behind API-key guards.

> **PyTorch → Keras:** `SentenceTransformer(...)` loads a PyTorch encoder and `.encode(...)` returns dense embeddings. A TensorFlow equivalent requires `TFAutoModel` or a Keras/TF-Hub encoder plus explicit pooling and normalization.

In [ ]:
# Embedding model + hybrid retriever.
print(f"Loading {EMBEDDING_MODEL_NAME} on {MODEL_DEVICE}...")
EMBED = SentenceTransformer(EMBEDDING_MODEL_NAME, device=MODEL_DEVICE)

# Encode every document once at index-build time; queries reuse this matrix.
DOC_EMBS = EMBED.encode(DOCS, show_progress_bar=False)

_tok_docs = [re.sub(r"[^\w\s]", "", document.lower()).split() for document in DOCS]
BM25 = BM25Okapi(_tok_docs)
print(f"  [ok]  model loaded, {len(DOCS)} documents encoded")


def retrieve(question, top_k=3):
    """Fuse semantic and BM25 rankings with reciprocal rank fusion."""
    query_embedding = EMBED.encode([question], show_progress_bar=False)
    semantic_scores = cosine_similarity(query_embedding, DOC_EMBS)[0]
    semantic_ranks = np.argsort(semantic_scores)[::-1]

    query_tokens = re.sub(r"[^\w\s]", "", question.lower()).split()
    lexical_scores = BM25.get_scores(query_tokens)
    lexical_ranks = np.argsort(lexical_scores)[::-1]

    rrf = {}
    for rank, index in enumerate(semantic_ranks, 1):
        rrf[index] = rrf.get(index, 0) + 1 / (60 + rank)
    for rank, index in enumerate(lexical_ranks, 1):
        rrf[index] = rrf.get(index, 0) + 1 / (60 + rank)

    ranked = sorted(rrf, key=lambda index: rrf[index], reverse=True)[:top_k]
    return [DOCS[index] for index in ranked], ranked


def generate(question, context_docs):
    """Extract the context sentence with highest embedding similarity to the question."""
    context = " ".join(context_docs)
    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+", context)
        if sentence.strip()
    ]
    query_embedding = EMBED.encode([question], show_progress_bar=False)
    sentence_embeddings = EMBED.encode(sentences, show_progress_bar=False)
    similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]
    return sentences[int(np.argmax(similarities))]


def rag_bot(question, top_k=3):
    """Retrieve context, then return the best-matching sentence as the answer."""
    docs, indices = retrieve(question, top_k=top_k)
    answer = generate(question, docs)
    return {"answer": answer, "retrieved_docs": docs, "retrieved_idxs": indices}


q0 = TEST_CASES[0]["question"]
r0 = rag_bot(q0)
print(f"\nrunning question: {q0!r}")
print(f"retrieved indices: {r0['retrieved_idxs']}")
print(f"generated answer:  {r0['answer']}")

### Code Walkthrough: Embedding Model, Hybrid Retriever, and RAG Bot

**What just ran — 5 key patterns:**

---

**`EMBED.encode(DOCS, show_progress_bar=False)` → `DOC_EMBS` — encode once, reuse always**
All 14 document embeddings are computed at index-build time and stored in `DOC_EMBS`. Every subsequent `retrieve()` call encodes only the query (1 vector) and then runs `cosine_similarity` against the pre-computed matrix — no re-encoding of documents at query time. This is the fundamental efficiency contract of dense retrieval: pay the embedding cost once, amortise it across every query.

---

**`retrieve(question, top_k=3)` — RRF fusion of semantic and BM25 rankings**
The function computes two independent ranked lists: semantic (cosine similarity to `DOC_EMBS`) and lexical (BM25 token scores). It accumulates `1/(60 + rank)` for each method into a shared RRF dict. Documents that rank high in both lists collect larger combined scores than documents that rank high in only one. The final `sorted(..., reverse=True)[:top_k]` returns the top-k document indices by merged score.

---

**`generate(question, context_docs)` — extractive answer, not generative**
This function is not calling an LLM. It splits the concatenated context into sentences, encodes each alongside the question, then returns the single sentence with the highest cosine similarity to the question. It returns **only that sentence** — not the prompt, not the full context. This makes the output deterministic and allows precise evaluation of what the "generator" contributed vs what the retriever retrieved.

---

**`rag_bot(question, top_k=3)` — thin orchestrator returning three evaluation handles**
`rag_bot` chains `retrieve → generate` and returns `{"answer", "retrieved_docs", "retrieved_idxs"}`. The three-field return is deliberate: each evaluation metric in the notebook needs a different field — `retrieved_docs` for retrieval relevance and groundedness, `answer` for answer relevance and ROUGE-L, `retrieved_idxs` for recall@1 against `gold_doc_idx`.

---

**Verification print block — smoke-test on the running question**
The final lines run `rag_bot` on `TEST_CASES[0]` and print the retrieved indices and extracted answer. This confirms the retriever grabbed the expected document (index 1, the ReAct doc) before the notebook proceeds to dissect four failure modes that depend on this same pipeline.

> **Shape/API note:** `retrieve` returns `([str, ...], [int, ...])` — document strings and their indices. `generate` receives the string list and returns a single `str`. Downstream cells always unpack via `result["answer"]`, `result["retrieved_docs"]`, and `result["retrieved_idxs"]`.


---

## Part 1 — The Four Silent Failures of RAG

**Riverside's question for this section:** before we build a single metric, can a person even tell
a wrong answer from a right one just by reading it?

**The common assumption:** if a RAG answer reads fluently and states its facts confidently, it's
probably correct. This section proves that assumption false in four independent ways at once — a
fluent, confident-sounding answer can fail for four completely different reasons, and none of them
show up as a stutter, a hedge, or bad grammar.

A RAG system performs two sequential operations:

$$\text{answer} = \text{Generate}\bigl(q,\; \text{Retrieve}(q)\bigr)$$

Either component can fail independently. The combination produces four distinct
failure modes that are indistinguishable from a correct answer without measurement.

**A taxonomy by cause:**

|                       | Generator faithful to context | Generator ignores context |
| --------------------- | ----------------------------- | ------------------------- |
| **Retriever correct** | Correct answer                | Hallucination             |
| **Retriever wrong**   | Coherent wrong answer         | Double failure            |

"Coherent wrong answer" is the hardest to detect by eye: the generator faithfully
summarises the retrieved context, the prose is fluent, and the document it cites
is real — but the wrong one. An evaluator reading only the answer has no signal.

We construct one example of each failure mode and carry them through every metric
in Parts 2-5 to prove that each failure type leaves a different fingerprint.


In [ ]:
#  Four failure modes on the running question
Q_THREAD = TEST_CASES[0]["question"]  # "How does ReAct combine reasoning and acting?"
REF = TEST_CASES[0]["reference"]
GOLD_CTX = DOCS[1]  # the ReAct document

CASES = {
    "Correct": {
        "answer": "ReAct interleaves thought and action steps, using tools like Wikipedia search to gather information while reasoning.",
        "context": GOLD_CTX,
        "note": "retriever correct, generator faithful",
    },
    "Coherent wrong": {
        "answer": "LoRA adds small trainable matrices to frozen weights, reducing tunable parameters by 10-100x.",
        "context": DOCS[7],  # LoRA document — wrong retrieval
        "note": "retriever wrong, generator faithful to wrong context",
    },
    "Hallucination": {
        "answer": "ReAct uses a neural circuit-breaker and quantum entanglement to synchronise reasoning heads across GPUs in real time.",
        "context": GOLD_CTX,  # correct context, but generator ignores it
        "note": "retriever correct, generator ignores context",
    },
    "Off-topic": {
        "answer": "Fine-tuning adapts a pretrained model to a specific task by continuing training on curated data.",
        "context": GOLD_CTX,
        "note": "retriever correct, answer about wrong topic",
    },
}

print(f"Question: {Q_THREAD!r}\n")
print(f"Reference answer: {REF!r}\n")
print("Four failure cases:")
for label, c in CASES.items():
    print(f"\n  [{label}]  ({c['note']})")
    print(f"  answer: {c['answer'][:90]}...")
    print(f"  context src: {c['context'][:60]}...")

#### What just happened — and what's missing

All four answers are grammatically correct English. A human reading them without the
reference answer and without the source documents cannot reliably sort them. This is
why manual spot-checking does not scale.

What we need: four metrics, each designed to catch one row or column in the failure
table:

- **Context Recall and Precision** (Part 2) — catches the "Retriever wrong" column
- **Groundedness** (Part 3) — catches the "Generator ignores context" row
- **Answer Relevance** (Part 4) — catches the off-topic answer
- **Correctness / ROUGE-L** (Part 5) — catches incomplete or wrong content

**Predict before you run Part 2:** the keyword overlap between the question
"How does ReAct combine reasoning and acting?" and the LoRA document is nonzero
(both use "model", "parameters"). Will a simple keyword-count metric flag the
wrong retrieval, or will it miss it?


---

## Part 2 — Retrieval Quality: Context Recall and Context Precision

**Riverside's question for this section:** when the assistant answers a catalog question, did it
actually pull up the right chapter — or just something that shares a few words with the question?

Retrieval quality has two complementary faces, familiar from information retrieval:

$$\text{Context Recall@k} = \frac{|D_{\text{ret}} \cap D_{\text{gold}}|}{|D_{\text{gold}}|}$$

$$\text{Context Precision@k} = \frac{|D_{\text{ret}} \cap D_{\text{gold}}|}{|D_{\text{ret}}|}$$

**Recall** asks: of all documents the answer requires, how many did we actually
retrieve? **Precision** asks: of all documents we retrieved, how many are actually
required?

The asymmetry matters. A retriever that dumps the entire corpus achieves perfect
recall at zero precision. A retriever that returns exactly one correct document
achieves perfect precision at potentially low recall if multiple documents are needed.

In the single-document evaluation we use here (each question has exactly one gold
document), both collapse to a binary 0/1 at rank 1. The interesting signal comes
from the continuous proxy: **average cosine similarity** between the query embedding
and the retrieved document embeddings.


In [ ]:
#  Attempt 1: keyword overlap as a retrieval quality proxy
#
# The simplest idea: count how many query tokens appear in the retrieved context.
# If the count is high, retrieval was probably on-topic.


def keyword_overlap_score(question, context):

    # Fraction of query content words found in context
    stop = {"a", "an", "the", "is", "are", "how", "does", "what", "which", "can", "do"}
    q_words = {w for w in question.lower().split() if w not in stop}
    c_words = set(context.lower().split())
    if not q_words:
        return 0.0
    return len(q_words & c_words) / len(q_words)


print("Keyword overlap score — good vs wrong retrieval:\n")
print(f"{'Case':<20} {'Overlap':>8}  Context preview")
print("-" * 80)
for label, c in CASES.items():
    score = keyword_overlap_score(Q_THREAD, c["context"])
    print(f"{label:<20} {score:>8.3f}  {c['context'][:60]}...")

print()
print("Problem: LoRA document contains 'model', 'parameters', 'weights'.")
print("Those words also appear in the ReAct question.  Keyword overlap is 0.3+")
print("even for the WRONG retrieval.  The metric cannot reliably separate them.")
print()
print("  -> naive keyword overlap is insufficient.  We need semantic similarity.")

> **PyTorch → Keras:** `EMBED.encode(...)` inside `retrieval_relevance` below runs the same PyTorch forward pass as before to embed the query and retrieved documents, then scores them with `sklearn`'s `cosine_similarity`. **Keras/TF equivalent:** call the loaded TF encoder on the batch of strings and compute similarity with `tf.keras.losses.CosineSimilarity` or `tf.linalg.l2_normalize(...)` plus a dot product — only the encoder call changes, the scoring math is framework-agnostic.


In [ ]:
#  Attempt 2: embedding cosine similarity
#
# Represent both query and each retrieved document as dense vectors.
# Cosine of the angle between them measures semantic alignment, not keyword overlap.
#
# For a set of retrieved documents D_ret the score is:
#
#   RetRel(q, D_ret) = (1 / |D_ret|) * sum_{d in D_ret} cos(e_q, e_d)


def context_recall_at1(question, retrieved_idxs, gold_doc_idx):

    # Binary: did we retrieve the gold document at top-1?
    return 1.0 if (retrieved_idxs[0] == gold_doc_idx) else 0.0


def retrieval_relevance(question, retrieved_docs):

    # Continuous proxy: average cosine similarity between query and retrieved docs
    if not retrieved_docs:
        return 0.0
    q_emb = EMBED.encode([question], show_progress_bar=False)
    d_embs = EMBED.encode(retrieved_docs, show_progress_bar=False)
    sims = cosine_similarity(q_emb, d_embs)[0]
    return float(np.mean(sims))


# Keyword overlap vs embedding similarity - side by side
print(f"{'Case':<20} {'Keyword':>8} {'Embedding':>10}  Separation?")
print("-" * 60)
for label, c in CASES.items():
    kw = keyword_overlap_score(Q_THREAD, c["context"])
    emb = retrieval_relevance(Q_THREAD, [c["context"]])
    sep = (
        "ok"
        if (label == "Correct" and emb > 0.6) or (label != "Correct" and emb < 0.4)
        else "?"
    )
    print(f"{label:<20} {kw:>8.3f} {emb:>10.3f}  {sep}")

# Score the standard RAG bot on all test questions
print(f"\nStandard RAG bot - retrieval relevance on all {len(TEST_CASES)} queries:\n")
rr_scores = []
recall_scores = []
for tc in TEST_CASES:
    result = rag_bot(tc["question"])
    rr = retrieval_relevance(tc["question"], result["retrieved_docs"])
    rc = context_recall_at1(
        tc["question"], result["retrieved_idxs"], tc["gold_doc_idx"]
    )
    rr_scores.append(rr)
    recall_scores.append(rc)
    hit = "[hit]" if rc == 1.0 else "[MISS]"
    print(f"  Q: {tc['question'][:50]:<50}  RR={rr:.3f}  recall@1={rc:.0f} {hit}")

print(f"\nmean retrieval relevance: {np.mean(rr_scores):.3f}")
print(f"mean recall@1:            {np.mean(recall_scores):.3f}")

In [ ]:
#  Retrieval quality: keyword vs embedding, good vs bad retrieval
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: keyword vs embedding for the four failure cases
labels = list(CASES.keys())
kw_vals = [keyword_overlap_score(Q_THREAD, c["context"]) for c in CASES.values()]
em_vals = [retrieval_relevance(Q_THREAD, [c["context"]]) for c in CASES.values()]
x = np.arange(len(labels))
w = 0.35
axes[0].bar(
    x - w / 2, kw_vals, w, label="Keyword overlap", color="steelblue", alpha=0.8
)
axes[0].bar(
    x + w / 2, em_vals, w, label="Embedding cosine", color="seagreen", alpha=0.8
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=15, ha="right")
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 1)
axes[0].set_title("Keyword vs Embedding Similarity\nfor the Four Failure Cases")
axes[0].legend()
axes[0].axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.7)

# Right: retrieval relevance across the complete labeled fixture
short_q = [f"Q{i+1}" for i in range(len(TEST_CASES))]
cols = ["seagreen" if s >= 0.55 else "coral" for s in rr_scores]
axes[1].barh(short_q[::-1], rr_scores[::-1], color=cols[::-1], alpha=0.85)
axes[1].axvline(0.55, color="gray", linestyle="--", linewidth=1)
axes[1].set_xlabel("Retrieval Relevance")
axes[1].set_title("Retrieval Relevance per Query\n(standard hybrid retriever)")
axes[1].set_xlim(0, 1)
axes[1].legend(
    handles=[
        Patch(
            facecolor="seagreen",
            edgecolor="black",
            label="Strong retrieval (>= 0.55)",
        ),
        Patch(facecolor="coral", edgecolor="black", label="Weak retrieval (< 0.55)"),
    ],
    loc="lower right",
    fontsize=8,
)

plt.suptitle("Part 2 - Retrieval Quality", fontweight="bold")
plt.tight_layout()
plt.show()

for i, tc in enumerate(TEST_CASES, 1):
    print(f"Q{i}: {tc['question']}")

#### Common Pitfall: Keyword Overlap Misses Paraphrased Matches

**Bad:** Score retrieval quality by counting shared words between the question and the retrieved
document.
**Good:** Score retrieval quality by embedding cosine similarity, which captures meaning instead of
exact wording.

**Why it matters:** a retrieved document can be exactly the right one and still share almost no
vocabulary with the question if it's phrased differently than the source text (e.g. asking "how do
agents alternate between thinking and doing" about a document that says "interleaves thought and
action steps"). Keyword overlap would score that correct retrieval as a near-miss, sending an
engineer to "fix" a retriever that was never broken.

**Quick Health Check:** run keyword overlap and embedding cosine on a paraphrase of the running
question that shares almost no vocabulary with the source document, and confirm embedding
similarity still recognises it as the right document while keyword overlap collapses.


In [ ]:
#  Quick health check: keyword overlap vs embedding on a real paraphrase
# Same intent as Q_THREAD, but reworded to share almost no vocabulary with the
# ReAct document ("interleaves thought and action steps... tools such as...").
paraphrased_question = "In what way does an agent alternate between thinking and doing?"

kw_orig = keyword_overlap_score(Q_THREAD, GOLD_CTX)
emb_orig = retrieval_relevance(Q_THREAD, [GOLD_CTX])
kw_para = keyword_overlap_score(paraphrased_question, GOLD_CTX)
emb_para = retrieval_relevance(paraphrased_question, [GOLD_CTX])

print(f"original question:    {Q_THREAD!r}")
print(f"paraphrased question: {paraphrased_question!r}")
print(f"context:               ReAct document\n")
print(f"{'Metric':<20} {'Original Q':>12} {'Paraphrased Q':>15}")
print("-" * 50)
print(f"{'Keyword overlap':<20} {kw_orig:>12.3f} {kw_para:>15.3f}")
print(f"{'Embedding cosine':<20} {emb_orig:>12.3f} {emb_para:>15.3f}")

if emb_para >= 0.5 and kw_para < kw_orig:
    print(
        f"\n  -> health check confirmed: embedding cosine stays high ({emb_para:.3f}) on the"
    )
    print(
        f"     paraphrase while keyword overlap drops ({kw_para:.3f}) -- exactly the gap"
    )
    print(f"     this metric exists to catch.")
else:
    print(f"\n  -> scores did not separate as expected on this paraphrase")
    print(
        f"     (keyword={kw_para:.3f}, embedding={emb_para:.3f}) -- inspect before trusting it."
    )

In [ ]:
#  Your turn — retrieval quality
# Change my_question to any topic.  A question outside the knowledge base should
# score near 0 on retrieval relevance; a question matching a document closely
# should score above 0.7.

my_question = "What is chain-of-thought prompting?"  # change here

docs, idxs = retrieve(my_question, top_k=3)
rr = retrieval_relevance(my_question, docs)
kw = keyword_overlap_score(my_question, " ".join(docs))

print(f"question:          {my_question!r}")
print(f"retrieved indices: {idxs}")
print(f"retrieval relevance (embedding): {rr:.3f}")
print(f"keyword overlap (naive):         {kw:.3f}")
for d in docs:
    print(f"  - {d[:85]}...")

verdict = (
    "strong retrieval" if rr >= 0.6 else "weak retrieval" if rr < 0.4 else "borderline"
)
print(f"\n  -> {verdict} (embedding cosine = {rr:.3f})")

#### What just happened — and what's missing

The comparison showed that keyword overlap cannot separate the "Correct" case from
the "Coherent wrong" case: the LoRA document shares enough vocabulary with the
question to score 0.3+ on keyword overlap. Embedding cosine puts them 0.4 apart.

But look at what embedding similarity cannot catch: the "Hallucination" case has
**the same context** as the correct case. Both score identically on retrieval relevance.
The metric tells us nothing about what the generator does with the retrieved text.

**Predict before you run Part 3:** the hallucinated answer
("ReAct uses quantum entanglement to synchronise reasoning heads") has some words
that do appear in the ReAct document ("reasoning", "heads"). Token recall will be
nonzero. Will the hallucination score above or below 0.3 on token recall?


---

## Part 3 — Groundedness: Does Every Claim Trace Back to the Context?

**Riverside's question for this section:** even when the assistant retrieves the right chapter, how
do we know it didn't just make something up instead of using it?

**Groundedness** (also called faithfulness) asks whether every assertion in the
generated answer is entailed by the retrieved documents.

The production definition uses natural language inference (NLI): a claim is
_grounded_ if the context logically entails it. Without an NLI model, we use token
recall as a tractable proxy. For answer $a$ and context $C$:

$$\text{Ground}(a, C) = \frac{|\text{tok}(a) \cap \text{tok}(C)|}{|\text{tok}(a)|}$$

where $\text{tok}(\cdot)$ strips punctuation, lowercases, and removes stop words.

**Why this works:** a grounded claim borrows its nouns, verbs, and technical terms
from the context. An invented claim ("quantum entanglement") must introduce new
vocabulary — those tokens have no counterpart in $C$, so the recall ratio drops.

**Why this breaks:** two failure modes remain even after token recall is high:

1. **Synonym hallucination** — the generator uses context vocabulary in a sentence
   with an opposite meaning ("ReAct does _not_ use Wikipedia search").
2. **Correct-but-ungrounded facts** — the answer is factually true but the source
   document didn't assert it, so there is no entailment, only coincidence.

Both require an NLI-capable judge. Token recall is a cheap first pass.


### Predict before you run

Rank the four answers from highest to lowest groundedness token recall before
running the token-recall groundedness cell further down.

The context for the "Correct", "Hallucination", and "Off-topic" cases is the
ReAct document (Doc 2):

> _"ReAct is an agent framework that interleaves thought and action steps,
> using tools such as Wikipedia search or a calculator."_

The context for the "Coherent wrong" case is the LoRA document (Doc 8).

Key question: the hallucinated answer contains the words "reasoning" and "heads"
which do appear in the context. Will it score above or below 0.25?


In [ ]:
#  Groundedness: token recall against context

_STOP = frozenset(
    {
        "a",
        "an",
        "the",
        "is",
        "are",
        "was",
        "were",
        "be",
        "been",
        "being",
        "to",
        "of",
        "in",
        "on",
        "at",
        "by",
        "for",
        "with",
        "from",
        "and",
        "or",
        "its",
        "their",
        "this",
        "that",
        "it",
        "we",
        "they",
        "i",
        "you",
        "not",
        "but",
    }
)


def tokenize(text):

    # lowercase, strip punctuation, drop stop words and single-char tokens
    tokens = re.sub(r"[^\w\s]", "", text.lower()).split()
    return {t for t in tokens if t not in _STOP and len(t) > 2}


def groundedness(answer, context):

    # Token recall: fraction of meaningful answer tokens present in context
    ans_tok = tokenize(answer)
    ctx_tok = tokenize(context)
    if not ans_tok:
        return 0.0
    return len(ans_tok & ctx_tok) / len(ans_tok)


print(f"{'Case':<20} {'Score':>7}  Shared                Missing")
print("-" * 90)

g_scores = {}

# Score groundedness for each failure case, plus which tokens matched/were missing
for label, c in CASES.items():
    score = groundedness(c["answer"], c["context"])
    g_scores[label] = score
    shared = sorted(tokenize(c["answer"]) & tokenize(c["context"]))[:5]
    missing = sorted(tokenize(c["answer"]) - tokenize(c["context"]))[:5]
    flag = "  <- hallucination" if score < 0.25 else ""
    print(f"{label:<20} {score:>7.3f}  {shared}  /  {missing}{flag}")

# Compare to the predict target
hall_score = g_scores["Hallucination"]
print(f"\nPrediction check:")
print(f"  Hallucination scored {hall_score:.3f}")
result_str = "above" if hall_score > 0.25 else "below"
print(f"  -> {result_str} 0.25 as predicted? see score above.")
print()
print(f"Correct answer scored {g_scores['Correct']:.3f}")
print(f"Gap (Correct - Hallucination) = {g_scores['Correct'] - hall_score:.3f}")

In [ ]:
#  Groundedness visualisation — bar chart and token overlap heatmap
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: groundedness per failure case
vals = list(g_scores.values())
lbls = list(g_scores.keys())
bcols = ["seagreen" if v >= 0.4 else "coral" for v in vals]
axes[0].bar(range(len(lbls)), vals, color=bcols, alpha=0.85)
axes[0].axhline(0.4, color="gray", linestyle="--", linewidth=1)
axes[0].set_xticks(range(len(lbls)))
axes[0].set_xticklabels(lbls, rotation=15, ha="right")
axes[0].set_ylabel("Groundedness (token recall)")
axes[0].set_ylim(0, 1)
axes[0].set_title("Groundedness per Failure Case")
axes[0].legend(
    handles=[
        Patch(facecolor="seagreen", edgecolor="black", label="Grounded (\u2265 0.4)"),
        Patch(facecolor="coral", edgecolor="black", label="Ungrounded (< 0.4)"),
    ],
    loc="upper right",
    fontsize=8,
)


# Right: token overlap heatmap — good vs hallucinated
def _overlap_mat(answer, context, n=10):

    # Binary match matrix: 1 where an answer token equals a context token, else 0
    atoks = list(dict.fromkeys(tokenize(answer)))[:n]
    ctoks = list(dict.fromkeys(tokenize(context)))[:n]
    mat = np.array([[1 if a == c else 0 for c in ctoks] for a in atoks])
    return mat, atoks, ctoks


m_g, ag, cg = _overlap_mat(CASES["Correct"]["answer"], GOLD_CTX)
m_h, ah, ch = _overlap_mat(CASES["Hallucination"]["answer"], GOLD_CTX)

# Stack the correct and hallucinated rows into one matrix, separated by a blank row
ctx_tok_union = list(dict.fromkeys(cg + ch))[:12]
all_rows = ag + ["--"] + ah
combined = np.zeros((len(all_rows), len(ctx_tok_union)))

# Fill in the "Correct" answer's token matches against the shared context vocabulary
for i, a in enumerate(ag):
    for j, c in enumerate(ctx_tok_union):
        combined[i, j] = 1 if a == c else 0

# Fill in the "Hallucination" answer's token matches, offset past the blank separator row
for i, a in enumerate(ah):
    row = len(ag) + 1 + i
    if row < combined.shape[0]:
        for j, c in enumerate(ctx_tok_union):
            combined[row, j] = 1 if a == c else 0

sns.heatmap(
    combined,
    ax=axes[1],
    cmap="YlGn",
    cbar=False,
    xticklabels=ctx_tok_union,
    yticklabels=all_rows,
    linewidths=0.4,
    linecolor="lightgray",
)
axes[1].set_title(
    "Token Overlap: Correct (top) vs Hallucinated (bottom)\n"
    "green = token found in ReAct context"
)
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(
    handles=[
        Patch(facecolor="#238b45", edgecolor="black", label="Token found in context"),
        Patch(facecolor="white", edgecolor="black", label="Token not found"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.32),
    ncol=2,
    fontsize=8,
)

plt.suptitle("Part 3 — Groundedness", fontweight="bold")
plt.tight_layout()
plt.show()

print("Bottom half (hallucination): introduced technical-sounding tokens")
print("absent from the context.  Token recall drops below 0.25 regardless")
print("of surface fluency.")

### Code Walkthrough: Groundedness — Bar Chart and Token Overlap Heatmap

**What just ran — 3 key patterns:**

---

**Colour-coded bar chart — green/coral at 0.4 threshold**
Each bar is coloured green if `score >= 0.4` and coral otherwise. The dashed `axhline(0.4)` makes the cut-off visible across all cases so the reader immediately sees which failure modes fall below "most answer tokens grounded in context." The threshold is intentionally round — it is a diagnostic aid, not a calibrated production SLA.

---

**`_overlap_mat(answer, context, n)` — sparse binary token-match matrix**
The helper extracts up to `n` unique stop-word-free tokens from both the answer and the context using `tokenize()`. It returns a binary matrix where `mat[i][j] = 1` if answer token `i` equals context token `j`. The `sns.heatmap` of this matrix makes token presence or absence concrete: rows where no cell is green are "missing tokens" that drove the groundedness score down for the hallucinated answer.

---

**Stacked layout — Correct (top) vs Hallucination (bottom) in one heatmap**
`combined` concatenates the correct-answer rows, a blank separator row, and the hallucinated-answer rows into a single matrix. This stacking makes the contrast visible without requiring side-by-side subplots: the top half should be mostly green (grounded), and the bottom half mostly empty (ungrounded), matching the bar-chart verdict directly to the left.

> **Shape/API note:** `combined` is `(len(ag) + 1 + len(ah)) × len(ctx_tok_union)`. The `+1` is the blank separator row. `ctx_tok_union` de-duplicates context tokens across both cases so both halves share the same column vocabulary for direct comparison.


In [ ]:
#  Where groundedness breaks: synonym hallucination
#
# Token recall misses cases where the generator uses context vocabulary
# with reversed or distorted meaning.  Construct a minimal adversarial case.

adversarial = (
    "ReAct does NOT interleave thought and action steps and avoids all tool use."
)
correct_ans = (
    "ReAct interleaves thought and action steps, using tools like Wikipedia search."
)

g_adv = groundedness(adversarial, GOLD_CTX)
g_corr = groundedness(correct_ans, GOLD_CTX)

print("Token recall cannot detect negation or distortion:\n")
print(f"  Correct:     score = {g_corr:.3f}")
print(f"    '{correct_ans[:70]}'")
print(f"\n  Adversarial: score = {g_adv:.3f}")
print(f"    '{adversarial[:70]}'")
print()
print("Both answers use the same context tokens ('interleave', 'thought', 'action',")
print("'tool', 'steps').  Token recall sees them as equally grounded.")
print()
print("  -> This is the boundary where token recall stops working and an NLI judge")
print("     (or LLM-as-judge) becomes necessary.  Part 7 addresses this.")

#### Common Pitfall: Groundedness Is Gameable by Copying Irrelevant Context Verbatim

**Bad:** Reward high token recall regardless of whether the copied context actually answers the
question asked.
**Good:** Always read groundedness together with answer relevance — a response can be nearly 100%
grounded in the retrieved text and still not address the question at all.

**Why it matters:** the cheapest way to score well on token recall is to copy long stretches of the
retrieved context verbatim into the answer. That maximises groundedness without requiring the
generator to understand anything, and it's a real failure mode for generators tuned to "play it
safe" by quoting rather than answering.

**Quick Health Check:** construct an "answer" that is literally a copy-paste of the context — not an
actual answer to the question — and confirm it scores highly on groundedness alone, then confirm
answer relevance is what actually catches the problem.


In [ ]:
#  Quick health check: a verbatim-copy "answer" games groundedness alone
copy_paste_answer = GOLD_CTX  # literally the context, not a real answer to Q_THREAD

g_copy = groundedness(copy_paste_answer, GOLD_CTX)

# Answer relevance isn't defined until Part 4 -- compute the same cosine
# comparison inline here so this health check doesn't depend on later cells.
_q_emb_tmp = EMBED.encode([Q_THREAD], show_progress_bar=False)
_a_emb_tmp = EMBED.encode([copy_paste_answer], show_progress_bar=False)
ar_copy = float(cosine_similarity(_q_emb_tmp, _a_emb_tmp)[0, 0])

print(f"question: {Q_THREAD!r}")
print(f"'answer' (verbatim context copy): {copy_paste_answer!r}\n")
print(f"groundedness score (token recall): {g_copy:.3f}")
print(f"answer-relevance-style cosine:      {ar_copy:.3f}")

if g_copy > 0.9:
    print(
        f"\n  -> health check confirmed: a verbatim copy games groundedness to {g_copy:.3f}"
    )
    print(
        f"     (every token is 'grounded' by definition), which is why groundedness alone"
    )
    print(
        f"     must never be the only gate on a real answer -- pair it with answer relevance."
    )
else:
    print(
        f"\n  -> unexpected: verbatim copy scored only {g_copy:.3f} on groundedness -- inspect."
    )

In [ ]:
#  Your turn — groundedness
# Change my_answer to experiment with how wording affects the score.
# Try: (a) adding a sentence that invents a fact not in the context
#       (b) replacing a key term with a synonym not in the context

my_answer = "ReAct uses tool calls and iterative reasoning to complete complex tasks."  # change here
my_context = DOCS[1]  # the ReAct document

score = groundedness(my_answer, my_context)
shared = sorted(tokenize(my_answer) & tokenize(my_context))
missing = sorted(tokenize(my_answer) - tokenize(my_context))

print(f"answer:      {my_answer!r}")
print(f"context:     {my_context[:80]}...")
print(f"\nGroundedness: {score:.3f}")
print(f"shared tokens:  {shared}")
print(f"missing tokens: {missing}")
verdict = (
    "well grounded"
    if score >= 0.5
    else "poorly grounded" if score < 0.3 else "borderline"
)
print(f"  -> {verdict}")

#### What just happened — and what's missing

Token recall exposed the hallucination as expected: invented technical jargon has
zero overlap with the context, dragging the score below 0.2. The adversarial example
showed the hard boundary: a factually inverted answer using only context vocabulary
scores identically to a correct answer.

Notice what neither retrieval relevance nor groundedness can see: the off-topic
answer. It retrieved the correct document (high retrieval relevance) and is grounded
in that document (high groundedness), yet it answers a completely different question.

**Predict before you run Part 4:** the off-topic answer is about fine-tuning.
The question is about ReAct. Predict whether embedding cosine similarity between
the question and the off-topic answer will be above or below 0.35.
The gap between the correct and off-topic answers on this metric is the key
measurement of answer relevance.


---

## Part 4 — Answer Relevance: Does the Answer Address the Question?

**Riverside's question for this section:** if someone asks about the mystery novel's founding
families and gets back a fluent, well-grounded paragraph about a different book, is that a pass?

**Answer relevance** measures whether the generated answer responds to the actual
question — independent of whether it is correct or grounded. It requires no
reference answer and no context document; it only compares query to response:

$$\text{AnsRel}(q, a) = \cos(e_q, e_a) = \frac{e_q \cdot e_a}{\|e_q\| \|e_a\|}$$

**Why cosine, not dot product?** The dot product grows with vector magnitude,
which varies by sentence length. Cosine normalises by both magnitudes, measuring
only the _direction_ — two texts about the same topic point the same way in
embedding space regardless of how long they are.

**What it catches:** an answer about fine-tuning, when the question was about ReAct,
will have an embedding pointing toward the fine-tuning region of the space. The
cosine to the question vector will be low.

**What it misses:** a hallucinated answer that _sounds like_ it is about ReAct
(uses "reasoning", "acting", "agent") will embed close to the question even if its
factual content is entirely invented. Answer relevance and groundedness are
complementary, not redundant.


### Predict before you run

The question is about ReAct. Two answers are about ReAct (Correct, Hallucination)
and two are not (Coherent wrong is about LoRA, Off-topic is about fine-tuning).

Before running, predict:

1. Will the two on-topic answers (Correct and Hallucination) score similarly?
2. What is the expected gap between the Correct answer and the Off-topic answer?
   - Less than 0.10
   - Between 0.10 and 0.25
   - More than 0.25


> **PyTorch → Keras:** `answer_relevance` below again calls `EMBED.encode(...)` (PyTorch forward pass) to embed the question and generated answer, then measures cosine similarity between the two vectors. **Keras/TF equivalent:** the same TF encoder-call pattern as above, paired with `tf.keras.losses.CosineSimilarity(axis=1)` — note Keras' version returns the *negative* cosine by convention, so the sign needs flipping to match this notebook's score.


In [ ]:
#  Answer relevance: cosine between query and answer embeddings


def answer_relevance(question, answer):

    # Cosine similarity between query embedding and answer embedding
    q_emb = EMBED.encode([question], show_progress_bar=False)
    a_emb = EMBED.encode([answer], show_progress_bar=False)
    return float(cosine_similarity(q_emb, a_emb)[0, 0])


print(f"{'Case':<20} {'Ans Relevance':>14}  note")
print("-" * 70)

ar_scores = {}

# Score answer relevance for every hand-crafted failure case
for label, c in CASES.items():
    score = answer_relevance(Q_THREAD, c["answer"])
    ar_scores[label] = score
    flag = "  <- off-topic" if score < 0.35 else ""
    print(f"{label:<20} {score:>14.3f}{flag}")

# Measure the gap the prediction asked about
gap_offtopic = ar_scores["Correct"] - ar_scores["Off-topic"]
gap_halluc = ar_scores["Correct"] - ar_scores["Hallucination"]
print(f"\nGap (Correct - Off-topic):     {gap_offtopic:.3f}")
print(f"Gap (Correct - Hallucination):  {gap_halluc:.3f}")
print("\nPrediction check:")
if gap_offtopic > 0.25:
    print(f"  -> gap is {gap_offtopic:.3f} > 0.25 (large)")
elif gap_offtopic > 0.10:
    print(f"  -> gap is {gap_offtopic:.3f}: between 0.10 and 0.25 (medium)")
else:
    print(
        f"  -> gap is {gap_offtopic:.3f} < 0.10 (small -- less separation than expected)"
    )

# Score the standard RAG bot on the complete labeled fixture
print(f"\nAnswer relevance - standard RAG bot, all {len(TEST_CASES)} queries:\n")
ar_bot = []
for tc in TEST_CASES:
    result = rag_bot(tc["question"])
    ar = answer_relevance(tc["question"], result["answer"])
    ar_bot.append(ar)
    print(f"  Q: {tc['question'][:50]:<50}  AnsRel={ar:.3f}")

In [ ]:
#  Your turn — answer relevance
# Observe how semantic drift affects the score.
# Try: (a) an answer that starts on-topic and then wanders
#       (b) a one-word answer like "Yes" — what does embedding similarity show?

my_question = "What types of adversarial attacks target LLMs?"  # keep fixed
my_answer = "LLMs face jailbreaking and prompt injection, though LoRA helps alignment."  # change here

score = answer_relevance(my_question, my_answer)
focused = "Adversarial attacks on LLMs include jailbreaking, prompt injection, and token manipulation."
sf = answer_relevance(my_question, focused)

print(f"question:        {my_question!r}")
print(f"your answer:     {my_answer!r}")
print(f"focused answer:  {focused!r}")
print(f"\nyour answer relevance:    {score:.3f}")
print(f"focused answer relevance: {sf:.3f}")
print(f"delta (focused - yours):  {sf - score:+.3f}")
verdict = "closer than expected" if abs(sf - score) < 0.05 else "noticeably different"
print(f"\n  -> embeddings are {verdict}")

#### What just happened — and what's missing

Answer relevance cleanly separated the off-topic answer: fine-tuning and ReAct embed
into different regions of the semantic space. Importantly, this required no reference
answer — only the question and the system's output.

The hallucinated answer scored close to the correct answer on relevance. This is the
expected and correct behaviour: both are genuinely about ReAct at the semantic level.
Their pathology is different (invented facts vs correct facts), and that difference
only becomes visible when we compare against ground-truth content.

**Predict before you run Part 5:** the correct answer is a paraphrase of the
reference. It uses "tools like Wikipedia search" where the reference says "tools
such as Wikipedia search". If we count only exact word matches (the naive approach),
what will the score be for this paraphrase — 0.0, 0.3, or 0.6+?


---

## Part 5 — Correctness: ROUGE-L and the Ordering Problem

**Riverside's question for this section:** an editor phrases the reference answer one way and the
assistant phrases a correct answer another way — does our scoring punish it for not being a
word-for-word match?

**Correctness** measures whether the generated answer contains the same information
as a reference answer. It requires labeled data — a gold-standard response for each
question.

### Why exact match fails

The simplest correctness metric is Jaccard similarity over word sets:

$$\text{Jaccard}(a, r) = \frac{|\text{tok}(a) \cap \text{tok}(r)|}{|\text{tok}(a) \cup \text{tok}(r)|}$$

This treats "tools such as Wikipedia search" and "tools like Wikipedia search" as
partial matches. More importantly, it ignores word order — a shuffled reference
scores identically to the original.

### ROUGE-L: longest common subsequence

ROUGE-L rewards order-preserving coverage. The longest common subsequence (LCS)
between answer $a$ and reference $r$ is the longest sequence of words that appears
in both, in the same relative order, without requiring them to be adjacent.

$$\text{ROUGE-L}(a, r) = \frac{|\text{LCS}(a, r)|}{|r|}$$

This is recall-oriented: we measure what fraction of the reference is covered by
the answer. An answer that covers half the reference words in the correct order
scores 0.5 even if it uses no exact phrases.

**The DP formulation:** $\text{LCS}(a, r)$ is computed with standard dynamic
programming. Let $L[i][j]$ be the LCS length for $a[:i]$ and $r[:j]$:

$$L[i][j] = \begin{cases} L[i-1][j-1] + 1 & a[i] = r[j] \\ \max(L[i-1][j],\; L[i][j-1]) & \text{otherwise} \end{cases}$$


In [ ]:
#  Attempt 1: exact word match (Jaccard similarity)


def jaccard(answer, reference):

    # Fraction of tokens shared between answer and reference, out of their union
    a_tok = set(answer.lower().split())
    r_tok = set(reference.lower().split())
    if not (a_tok | r_tok):
        return 0.0
    return len(a_tok & r_tok) / len(a_tok | r_tok)


# A correct paraphrase that uses synonyms
paraphrase = "ReAct alternates between reasoning steps and actions, employing tools such as web search during the process."

print("Exact match (Jaccard) on a correct paraphrase:\n")
print(f"  reference:  {REF!r}")
print(f"  paraphrase: {paraphrase!r}")
print(f"  Jaccard:    {jaccard(paraphrase, REF):.3f}")
print()
print("The paraphrase is factually correct and semantically equivalent.")
print(
    "Jaccard penalises every synonym: 'alternates'/'interleaves', 'employing'/'using', etc."
)
print()
print(
    "  -> Jaccard treats synonyms as wrong.  We need order-preserving partial credit."
)

In [ ]:
#  ROUGE-L from scratch: LCS via dynamic programming


def lcs_length(a, b):

    # Standard DP: L[i][j] = LCS length for a[:i] and b[:j]
    m, n = len(a), len(b)
    L = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):

            # Extend the diagonal match, or carry forward the best neighboring cell
            if a[i - 1] == b[j - 1]:
                L[i][j] = L[i - 1][j - 1] + 1
            else:
                L[i][j] = max(L[i - 1][j], L[i][j - 1])
    return L[m][n]


def rouge_l(answer, reference):

    # Recall-oriented: LCS / reference length
    a_tok = answer.lower().split()
    r_tok = reference.lower().split()
    if not r_tok:
        return 0.0
    return lcs_length(a_tok, r_tok) / len(r_tok)


# Score all four failure cases
print(f"Reference: {REF!r}\n")
print(f"{'Case':<20} {'Jaccard':>8} {'ROUGE-L':>9}  note")
print("-" * 55)

rl_scores = {}
for label, c in CASES.items():
    j = jaccard(c["answer"], REF)
    rl = rouge_l(c["answer"], REF)
    rl_scores[label] = rl
    flag = "  <- low coverage" if rl < 0.20 else ""
    print(f"{label:<20} {j:>8.3f} {rl:>9.3f}{flag}")

# Show the paraphrase fix
rl_par = rouge_l(paraphrase, REF)
j_par = jaccard(paraphrase, REF)
print(
    f"\nParaphrase:          {j_par:>8.3f} {rl_par:>9.3f}  (correct paraphrase - ROUGE-L gives credit)"
)
print()
print("ROUGE-L gives the paraphrase partial credit via shared subsequence;")
print("Jaccard penalises every synonym. Gap = ", round(rl_par - j_par, 3))

# Score the complete labeled fixture with the standard RAG bot
print(f"\nROUGE-L - standard RAG bot, all {len(TEST_CASES)} queries:\n")
rl_bot = []
for tc in TEST_CASES:
    result = rag_bot(tc["question"])
    rl = rouge_l(result["answer"], tc["reference"])
    rl_bot.append(rl)
    print(f"  Q: {tc['question'][:50]:<50}  RL={rl:.3f}")

### Code Walkthrough: ROUGE-L via Dynamic Programming

**What just ran - a DP algorithm plus two separate applications of it:**

---

**`lcs_length(a, b)` - the classic longest-common-subsequence table**

`L` is an `(m+1) x (n+1)` table where `L[i][j]` holds the LCS length of `a[:i]` and `b[:j]`. Each cell either extends a diagonal match (`a[i-1] == b[j-1]`) or carries forward the best of the cell above or to the left. This is the textbook LCS recurrence from the formula shown above, with `L[m][n]` in the bottom-right corner holding the final answer.

> **Shape note:** `L` has one extra row and column versus `a`/`b` so that `L[0][*]` and `L[*][0]` can represent "compared against an empty sequence" (always 0) without a special case.

---

**`rouge_l(answer, reference)` - turning LCS length into a recall score**

Divides `lcs_length` by `len(r_tok)` (the reference length, not the answer length). This makes ROUGE-L recall-oriented rather than precision-oriented, a distinction the surrounding markdown calls out explicitly.

---

**Two separate applications of the same function, back to back**

The cell first scores `rouge_l` against the four hand-crafted `CASES`, a clean taxonomy demonstration using the same pattern as Part 3's groundedness cell. It then reruns the metric against `rag_bot`'s live output on all eight `TEST_CASES`, the system's actual current performance. Keeping both in one cell makes the contrast easy to read: the constructed examples show *why* ROUGE-L beats Jaccard, and the live scores show *what the real pipeline currently gets*.

#### Common Pitfall: Exact-Match Correctness Unfairly Scores a Valid Paraphrase as Near-Zero

**Bad:** Score correctness with exact word-set overlap (Jaccard) and treat a low score as "the
assistant got it wrong."
**Good:** Use an order-preserving, partial-credit metric like ROUGE-L (or an LLM judge) before
concluding an answer is factually wrong.

**Why it matters:** the paraphrase test above already showed Jaccard penalising every synonym in a
correct answer. If Riverside gated a release on exact-match correctness, a genuinely correct,
differently-worded answer would be flagged as a regression, and someone would spend an afternoon
"fixing" a bug that doesn't exist.

**Quick Health Check:** pick a pass/fail threshold and confirm ROUGE-L and Jaccard actually disagree
on the paraphrase at that threshold — if they don't disagree on this example, say so plainly rather
than forcing the point.


In [ ]:
#  Quick health check: does ROUGE-L flip a pass/fail decision Jaccard gets wrong?
THRESHOLD = 0.5

j_para = jaccard(paraphrase, REF)
rl_para = rouge_l(paraphrase, REF)

j_verdict = "PASS" if j_para >= THRESHOLD else "FAIL"
rl_verdict = "PASS" if rl_para >= THRESHOLD else "FAIL"

print(f"reference:  {REF!r}")
print(f"paraphrase: {paraphrase!r}  (factually correct, different wording)\n")
print(f"{'Metric':<12} {'Score':>7}  {'Verdict @ ' + str(THRESHOLD):>14}")
print("-" * 40)
print(f"{'Jaccard':<12} {j_para:>7.3f}  {j_verdict:>14}")
print(f"{'ROUGE-L':<12} {rl_para:>7.3f}  {rl_verdict:>14}")

if j_verdict != rl_verdict:
    print(
        f"\n  -> health check confirmed: exact-match would {j_verdict} a genuinely correct"
    )
    print(
        f"     paraphrase ({j_para:.3f}) that ROUGE-L correctly lets through ({rl_para:.3f})."
    )
else:
    print(
        f"\n  -> both metrics agree ({j_verdict}) on this example at threshold {THRESHOLD};"
    )
    print(
        f"     the {rl_para - j_para:.3f} gap between them is still the one to watch as"
    )
    print(f"     paraphrases drift further from the reference wording.")

#### ROUGE-L Still Has a Ceiling: Heavy Vocabulary Substitution

The paraphrase health check above fixed the _word-order_ problem: ROUGE-L credits a paraphrase
that keeps most of the reference's actual words, just reordered or lightly reworded. It does not
fix a harder case — a paraphrase that keeps the same _meaning_ while replacing almost every
content word with a synonym. ROUGE-L still measures literal token overlap inside a longest-common-
subsequence, so it has no way to recognize "employ" and "use," or "framework" and "system," as the
same idea. This is the same proxy boundary Part 3 hit with groundedness — a metric built on
surface tokens runs out of room exactly where meaning and wording diverge.


In [ ]:
#  Quick check: a heavy-synonym paraphrase still depresses ROUGE-L
# Reworded so aggressively that almost no content word survives, while the meaning
# (and factual content) stays identical to the reference.
heavy_paraphrase = "An agent framework merges cognitive steps with external actions by weaving them together, drawing on resources like an online encyclopedia lookup."

light_paraphrase_rl = rouge_l(
    paraphrase, REF
)  # the light reword from the health check above
heavy_paraphrase_rl = rouge_l(heavy_paraphrase, REF)  # near-total synonym replacement
heavy_paraphrase_emb = answer_relevance(Q_THREAD, heavy_paraphrase)

print(f"reference:          {REF!r}\n")
print(f"light paraphrase:   {paraphrase!r}")
print(f"  ROUGE-L: {light_paraphrase_rl:.3f}\n")
print(f"heavy paraphrase:   {heavy_paraphrase!r}")
print(f"  ROUGE-L: {heavy_paraphrase_rl:.3f}")
print(f"  answer relevance (embedding): {heavy_paraphrase_emb:.3f}")

print()
if heavy_paraphrase_rl < light_paraphrase_rl:
    print(
        f"  -> ROUGE-L drops further ({heavy_paraphrase_rl:.3f} vs {light_paraphrase_rl:.3f}) as"
    )
    print(
        f"     vocabulary overlap shrinks, even though the meaning hasn't changed -- confirmed by"
    )
    print(
        f"     the embedding-based relevance score staying high ({heavy_paraphrase_emb:.3f})."
    )
    print(
        f"     This is the exact boundary an embedding-based correctness metric (BERTScore) or an"
    )
    print(f"     LLM judge (Part 7) is built to cross.")
else:
    print(
        f"  -> scores did not separate as expected on this example -- inspect before trusting it."
    )

In [ ]:
#  All four metrics on the four failure cases — the fingerprint table
metric_data = {}

# Compute all four metrics for every hand-crafted failure case
for label, c in CASES.items():
    metric_data[label] = {
        "Retrieval Rel.": retrieval_relevance(Q_THREAD, [c["context"]]),
        "Groundedness": groundedness(c["answer"], c["context"]),
        "Answer Rel.": answer_relevance(Q_THREAD, c["answer"]),
        "ROUGE-L": rouge_l(c["answer"], REF),
    }

# Pivot into failure-case rows x metric columns for both the table and the chart below
df_fp = pd.DataFrame(metric_data).T
print("Metric fingerprint for each failure mode:\n")
print(df_fp.round(3).to_string())

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(df_fp.columns))
width = 0.18
colors = ["#4c9be8", "#56b356", "#e8934c", "#c65454"]

# Draw one bar group per failure case, offset by width so all four metrics sit side by side
for i, (label, row) in enumerate(df_fp.iterrows()):
    ax.bar(x + i * width, row.values, width, label=label, color=colors[i], alpha=0.85)
ax.axhline(0.4, color="gray", linestyle="--", linewidth=1, alpha=0.6)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_fp.columns)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.1)
ax.set_title(
    "All Four Metrics per Failure Mode\n" "Each failure type drops a different metric"
)
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

print("\nReading the chart:")
print("  Correct           -> all four metrics high")
print("  Coherent wrong    -> Retrieval Rel. drops; others moderate")
print("  Hallucination     -> Groundedness + ROUGE-L drop; Ans Rel. stays high")
print("  Off-topic         -> Answer Rel. + ROUGE-L drop; Groundedness may stay high")

### Code Walkthrough: Multi-Metric Fingerprint for All Four Failure Cases

**What just ran — 3 key patterns:**

---

**`metric_data` dict of dicts — structured multi-metric collection**
The outer loop over `CASES` reuses the same four metric functions (`retrieval_relevance`, `groundedness`, `answer_relevance`, `rouge_l`) and stores the scores in a nested dict. Calling `pd.DataFrame(metric_data).T` pivots the structure so rows are failure cases and columns are metrics — the exact shape needed for both the printed table and the bar chart that follows.

---

**Grouped bar chart — four cases, four colour bands, shared threshold line**
Each failure case gets one colour and one group of four bars. `width = 0.18` fits four groups without overlap. The dashed `axhline(0.4)` provides a shared visual reference: bars dropping below this line are the "failing dimensions" for each case, making the 2×2 failure taxonomy readable at a glance without reading individual values.

---

**"Reading the chart" print block — explicit taxonomy decoding**
The four lines at the bottom decode the chart back into the Part 1 taxonomy: Correct = all high, Coherent wrong = Retrieval Rel. drops, Hallucination = Groundedness + ROUGE-L drop, Off-topic = Answer Rel. + ROUGE-L drop. This explicit decoding is the pedagogical payoff: the chart is not decorative — it is proof that the four metrics are orthogonal and collectively sufficient to diagnose the failure type.

> **Note:** `metric_data` uses manually crafted answers for a clean taxonomy demonstration. `df_dash` in Part 6 uses live `rag_bot` output on the full test set. Both views are useful: `CASES` demonstrates the taxonomy cleanly; `df_dash` measures the real system's actual performance.


### Multi-Metric Comparison: All Four Metrics × All Four Failure Cases

The table below summarises which metric detects which failure mode. ↓ = metric drops for that case;  = stays high:

| Failure Case                   | Retrieval Rel. | Groundedness | Answer Rel. | ROUGE-L | Root Cause                                                    |
| ------------------------------ | -------------- | ------------ | ----------- | ------- | ------------------------------------------------------------- |
| Correct                        |               |             |            |        | No failure — all metrics high                                 |
| Coherent wrong (bad retrieval) | ↓              |  moderate   |  moderate  | ↓       | Wrong document retrieved; generator faithful to wrong context |
| Hallucination (invented facts) |               | ↓            |            | ↓       | Generator ignores retrieved context, invents content          |
| Off-topic (answer drifts)      |               |             | ↓           | ↓       | Generator answers a different question than what was asked    |

No single metric catches all four failures — they are orthogonal by design. The dual heatmap below renders the same data as a grid so you can see the score pattern at a glance rather than reading the table row by row.


In [ ]:
#  Multi-Metric Comparison Dual Heatmap
# Pattern 3 — failure case × metric heatmap: raw scores and pass/fail panels

metric_names_h = ["Retrieval Rel.", "Groundedness", "Answer Rel.", "ROUGE-L"]
case_names_h = list(CASES.keys())
n_cases_h, n_met = len(case_names_h), len(metric_names_h)

# Pre-fill with NaN so any un-computed cell would render as grey ("not available")
score_grid_h = np.full((n_cases_h, n_met), np.nan)

# Fill the failure-case x metric grid with all four metrics for every case
for ci, (label, c) in enumerate(CASES.items()):
    score_grid_h[ci, 0] = retrieval_relevance(Q_THREAD, [c["context"]])
    score_grid_h[ci, 1] = groundedness(c["answer"], c["context"])
    score_grid_h[ci, 2] = answer_relevance(Q_THREAD, c["answer"])
    score_grid_h[ci, 3] = rouge_l(c["answer"], REF)

# Pass/fail panel: threshold = 0.4 for first three metrics, 0.2 for ROUGE-L
thresholds_h = [0.4, 0.4, 0.4, 0.2]

# Derive a second grid: does each raw score clear its own metric's threshold?
pf_grid = np.array(
    [
        [score_grid_h[ci, mi] >= thresholds_h[mi] for mi in range(n_met)]
        for ci in range(n_cases_h)
    ]
).astype(float)

# Grey out any NaN cell instead of letting matplotlib guess a color for it
cmap_score_h = plt.cm.YlGn.copy()
cmap_score_h.set_bad(color="#d3d3d3")
cmap_pf_h = plt.cm.RdYlGn.copy()
cmap_pf_h.set_bad(color="#d3d3d3")

fig_mm, (ax_s, ax_p) = plt.subplots(1, 2, figsize=(14, 4))

# Draw both panels (raw scores, then pass/fail) with one loop over different data/colormaps
for ax, mat, cmap_u, title_u in [
    (ax_s, score_grid_h, cmap_score_h, "Raw Score (0–1)"),
    (ax_p, pf_grid, cmap_pf_h, "Pass (≥ threshold)"),
]:
    im = ax.imshow(mat, cmap=cmap_u, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(n_met))
    ax.set_xticklabels(metric_names_h, rotation=20, ha="right", fontsize=9)
    ax.set_yticks(range(n_cases_h))
    ax.set_yticklabels(case_names_h, fontsize=9)
    ax.set_title(f"Failure Case × Metric — {title_u}")

    # Annotate each cell with its numeric value or PASS/FAIL label
    for ci in range(n_cases_h):
        for mi in range(n_met):
            v = mat[ci, mi]
            txt = f"{v:.2f}" if title_u.startswith("Raw") else ("PASS" if v else "FAIL")
            colour = (
                "white"
                if (v > 0.65 or (not title_u.startswith("Raw") and v))
                else "black"
            )
            ax.text(mi, ci, txt, ha="center", va="center", fontsize=8, color=colour)
    fig_mm.colorbar(im, ax=ax)

plt.suptitle(
    "Dual Heatmap: RAG Failure Taxonomy × Evaluation Metric\n"
    "Each failure mode drops a different metric — no single metric catches all",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print("\nTaxonomy confirmed by the heatmap:")
print("  Retrieval Rel. drops for 'Coherent wrong'  — wrong document retrieved")
print("  Groundedness drops for 'Hallucination'     — generator invents content")
print("  Answer Rel. drops for 'Off-topic'          — generator drifts from question")
print(
    "  ROUGE-L drops for 'Hallucination' and 'Off-topic' — content mismatch with reference"
)

### Code Walkthrough: Dual Score / Pass-Fail Heatmap Grid

**What just ran — 3 patterns combined into one cell:**

---

**`score_grid_h` — an `np.full(..., np.nan)` grid, filled in a loop**

Rather than build a nested list and convert it, the cell pre-allocates a `(n_cases, n_metrics)`
array of `NaN` and fills each `[ci, mi]` cell by calling the four metric functions already defined
earlier in the notebook. Starting from `NaN` (not `0`) matters for the next step: any cell that
was never assigned stays `NaN` and renders as a distinct "missing" colour rather than a
misleading `0.0`.

---

**`pf_grid` — deriving a second grid from the first via per-metric thresholds**

`thresholds_h` holds one cutoff per metric (0.4 for the first three, 0.2 for ROUGE-L, matching
the thresholds already used in each metric's own section above). The list comprehension compares
every raw score against its column's threshold, producing a boolean pass/fail grid the same shape
as `score_grid_h` — this is what lets the right-hand panel show "PASS"/"FAIL" text instead of a
raw number, using the exact same layout as the left panel for a direct side-by-side read.

---

**Dual `imshow` loop with `cmap.set_bad()` and inline text annotations**

Both panels are drawn by the same `for ax, mat, cmap_u, title_u in [...]` loop rather than two
separate blocks of near-identical code. `cmap.set_bad(color="#d3d3d3")` is what makes a `NaN` cell
render as grey instead of throwing an error or defaulting to a colormap endpoint. The nested
`ax.text(...)` loop overlays either the raw score or a PASS/FAIL string on every cell, and each
panel gets its own `fig_mm.colorbar(im, ax=ax)` — this is the per-subplot legend requirement in
practice: a shared colorbar for two panels with different value meanings (raw score vs. boolean)
would be misleading, so each panel earns its own.


#### What just happened — and what's missing

The fingerprint table confirms the taxonomy from Part 1: each failure mode has a
distinct pattern across the four metrics. No single metric catches all four failures;
they are orthogonal by design.

ROUGE-L improved on Jaccard by giving the paraphrase partial credit through the LCS
formulation. The remaining gap — ROUGE-L penalising all synonyms — is where
embedding-based correctness metrics (and LLM-as-judge) outperform n-gram metrics.

**The dependency structure matters:** retrieval failure is upstream. When the
retriever returns the wrong document, the generator cannot possibly produce a
correct answer from that context alone. Fixing retrieval relevance is always the
higher-leverage action — it is a necessary (though not sufficient) precondition for
all other metrics to be meaningful.


---

## Part 6 — Composite Dashboard: Diagnosing the Bottleneck

**Riverside's question for this section:** when the assistant is clearly underperforming on some
questions, which piece do we actually go fix — the retriever, the generator, or the prompt?

A composite score by itself tells you how good the system is. The individual metrics
tell you _where_ to invest effort. The right diagnostic view is the per-query
breakdown, not the aggregate.

**Simple composite:** arithmetic mean of all four metrics:

$$\text{score}(q) = \frac{1}{4}\bigl(\text{RetRel} + \text{Ground} + \text{AnsRel} + \text{ROUGE-L}\bigr)$$

A low composite score could come from a bad retriever (fix the index or embedding
model), a hallucinating generator (add groundedness checks or constrain generation),
or a topic-drifting generator (improve prompting or fine-tune). Without the breakdown
you can't tell which.


In [ ]:
#  Per-query evaluation dashboard
dashboard_rows = []

# Run every metric against the live RAG bot's real output, per query, plus a composite mean
for tc in TEST_CASES:
    result = rag_bot(tc["question"])
    ctx = " ".join(result["retrieved_docs"])
    row = {
        "Query": tc["question"][:42] + "...",
        "RetRel": retrieval_relevance(tc["question"], result["retrieved_docs"]),
        "Groundedness": groundedness(result["answer"], ctx),
        "AnsRel": answer_relevance(tc["question"], result["answer"]),
        "ROUGE-L": rouge_l(result["answer"], tc["reference"]),
    }
    row["Composite"] = float(
        np.mean([row["RetRel"], row["Groundedness"], row["AnsRel"], row["ROUGE-L"]])
    )
    dashboard_rows.append(row)

df_dash = pd.DataFrame(dashboard_rows)
print("RAG Evaluation Dashboard — Standard Hybrid Bot\n")
print(df_dash.round(3).to_string(index=False))

metric_cols = ["RetRel", "Groundedness", "AnsRel", "ROUGE-L"]

# Find the weakest metric on average across all queries — that's the bottleneck to fix first
mean_per = df_dash[metric_cols].mean()
worst = mean_per.idxmin()
print(f"\nMean composite score:  {df_dash['Composite'].mean():.3f}")
print(f"Weakest metric:        {worst}  ({mean_per[worst]:.3f})")
print(f"\n  -> Fix {worst} first.  It is the biggest drag on the composite.")

In [ ]:
#  Per-query metric fingerprint — animated radar (FuncAnimation)
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

metric_cols = ["RetRel", "Groundedness", "AnsRel", "ROUGE-L"]
N = len(metric_cols)

# One evenly spaced angle per metric; repeat the first angle to close the polygon
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

n_q_radar = len(df_dash)
cmap_radar = plt.cm.tab10

fig_anim, ax_anim = plt.subplots(1, 1, figsize=(7, 7), subplot_kw={"polar": True})


def _update_radar(frame):

    # Redraw from scratch each frame, overlaying every query's polygon up to this one
    ax_anim.clear()
    ax_anim.set_xticks(angles[:-1])
    ax_anim.set_xticklabels(metric_cols, size=10)
    ax_anim.set_ylim(0, 1)
    ax_anim.set_title(f"Per-Query Metric Fingerprint — Q1 to Q{frame + 1}", pad=20)
    ideal = [1.0] * (N + 1)
    ax_anim.plot(angles, ideal, "k--", linewidth=0.8, alpha=0.2)

    # Plot every query's polygon so far, fading all but the current frame
    for i in range(frame + 1):
        row = df_dash.iloc[i]
        vals = [row[m] for m in metric_cols] + [row[metric_cols[0]]]
        alpha = 1.0 if i == frame else 0.35
        ax_anim.plot(
            angles,
            vals,
            "o-",
            linewidth=1.8,
            color=cmap_radar(i),
            alpha=alpha,
            label=f"Q{i+1}",
        )
        ax_anim.fill(angles, vals, alpha=0.06, color=cmap_radar(i))
    if frame == n_q_radar - 1:
        ax_anim.legend(loc="upper right", bbox_to_anchor=(1.45, 1.15), fontsize=9)


anim = FuncAnimation(
    fig_anim, _update_radar, frames=n_q_radar, interval=900, repeat=False
)
plt.close(fig_anim)

print("Animated per-query metric fingerprint — one query profile added per frame.")
print("Current frame shown opaque; earlier queries faded to 35%.")
print(
    "An asymmetric 'dent' in the profile marks the weakest component for that query.\n"
)
display(HTML(anim.to_jshtml(fps=6)))

#  Static mean profile vs ideal
# Average each metric across all queries for the static comparison panel
mean_per = df_dash[metric_cols].mean()
mean_v = [mean_per[m] for m in metric_cols] + [mean_per[metric_cols[0]]]
ideal = [1.0] * (N + 1)

fig_static, ax_static = plt.subplots(figsize=(6, 6), subplot_kw={"polar": True})
ax_static.plot(angles, ideal, "k--", linewidth=1, alpha=0.25, label="Ideal")
ax_static.fill(angles, ideal, alpha=0.04, color="gray")
ax_static.plot(angles, mean_v, "o-", linewidth=2.5, color="steelblue", label="Mean")
ax_static.fill(angles, mean_v, alpha=0.18, color="steelblue")
ax_static.set_xticks(angles[:-1])
ax_static.set_xticklabels(metric_cols, size=10)
ax_static.set_ylim(0, 1)
ax_static.set_title("Mean Profile vs Ideal", pad=20)
ax_static.legend(loc="upper right", bbox_to_anchor=(1.45, 1.15))
plt.suptitle("Part 6 — Composite Dashboard", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("The dent in the radar relative to the ideal circle marks the weakest")
print("component.  Fix the metric with the largest deficit first.")

### Code Walkthrough: Animated and Static Radar Charts

**What just ran - a `FuncAnimation` polar plot followed by a static comparison version:**

---

**Closing the polygon - `angles += angles[:1]`**

`np.linspace(0, 2*np.pi, N, endpoint=False)` gives `N` evenly spaced angles, one per metric, but a radar polygon needs its last point to connect back to the first to close the shape visually. Appending `angles[:1]`, and appending the first metric value in `_update_radar`, makes `ax.plot(angles, vals, ...)` draw a closed loop instead of a shape with one open edge.

---

**`_update_radar(frame)` - full `ax_anim.clear()` every frame, not incremental updates**

Unlike the bar-chart `FuncAnimation` pattern used earlier in this notebook, a polar radar chart with a growing number of overlaid queries is simpler to redraw from scratch each frame. `ax_anim.clear()` wipes the axes, then the loop replots every query up to the current frame and fades earlier queries. The legend is attached only on the final frame so it does not reflow on every tick.

---

**Static "Mean Profile vs Ideal" panel - the same closed-polygon technique, one snapshot**

The static chart directly beneath the animation reuses the same closing-the-loop technique, but plots only two series: the per-metric mean across all eight queries and an ideal reference polygon. Reading the two together is the point: the animation shows how the fingerprint accumulates query by query, and the static panel shows where it lands on average. The dent between the mean line and the ideal circle marks the metric most worth investigating first.

> **Rendering note:** `plt.close(fig_anim)` runs immediately after `FuncAnimation(...)` and before `display(HTML(anim.to_jshtml(fps=6)))`. This suppresses the duplicate static frame Jupyter would otherwise render.

#### What just happened — and what's missing

The dashboard converts four scalar metrics into a diagnostic signal: not "how good is
the system" but "which component is the bottleneck." The radar chart makes the
fingerprint spatial — a system with a bad retriever has a left-side dent; a
hallucinating generator has a bottom dent.

**The remaining limitation:** every metric in this notebook is a proxy. Each uses
word-level features or sentence-level embeddings to approximate a judgment that
requires reading comprehension. The proxies work well for obvious failures (wrong
retrieval, invented jargon); they break for subtle failures (synonym hallucination,
factual distortion with borrowed vocabulary).

Part 7 shows how the same metric _structure_ survives the move to production, with
only the scorer swapped from word overlap to an LLM NLI judge.


---

## Part 6b - Oracle Context: Hand the Generator the Right Page

The dashboard tells Riverside **which metric is weak**, but an end-to-end answer can still hide who caused the failure. A poor answer may come from two very different roadblocks:

1. the retriever handed the generator the wrong document;
2. the generator received the right document and still produced a weak answer.

The cleanest diagnostic is temporarily unfair: give the generator the gold document directly. For the running ReAct question, that means bypassing hybrid retrieval and handing `generate()` the known ReAct passage.

| What happens with gold context? | What Riverside learns |
| --- | --- |
| Groundedness or correctness jumps | The generator can use the right evidence; retrieval is the bottleneck |
| Scores barely move | Better retrieval alone will not help; inspect answer construction or the evaluator |
| Some questions jump and others do not | The system has more than one failure mode; inspect those slices separately |

This is an **oracle ablation**, not a production feature. Riverside cannot use a gold-document label at request time. The forced context removes one moving part so the team fixes the component that actually failed instead of tuning the whole pipeline by feel.

In [ ]:
# Compare the live end-to-end path with the same generator given the labeled gold document.
def score_answer_path(question, answer, context_docs, reference):
    context = " ".join(context_docs)
    scores = {
        "Groundedness": groundedness(answer, context),
        "AnsRel": answer_relevance(question, answer),
        "ROUGE-L": rouge_l(answer, reference),
    }
    scores["AnswerSignal"] = float(np.mean(list(scores.values())))
    return scores


oracle_rows = []
for case_index, test_case in enumerate(TEST_CASES, start=1):
    question = test_case["question"]
    end_to_end = rag_bot(question)
    gold_context = [DOCS[test_case["gold_doc_idx"]]]
    oracle_answer = generate(question, gold_context)

    end_scores = score_answer_path(
        question,
        end_to_end["answer"],
        end_to_end["retrieved_docs"],
        test_case["reference"],
    )
    oracle_scores = score_answer_path(
        question,
        oracle_answer,
        gold_context,
        test_case["reference"],
    )
    answer_signal_gain = oracle_scores["AnswerSignal"] - end_scores["AnswerSignal"]
    oracle_rows.append(
        {
            "Query": f"Q{case_index}",
            "GoldAt1": end_to_end["retrieved_idxs"][0] == test_case["gold_doc_idx"],
            "EndToEnd": end_scores["AnswerSignal"],
            "GoldContext": oracle_scores["AnswerSignal"],
            "Gain": answer_signal_gain,
            "LikelyBottleneck": (
                "retriever" if answer_signal_gain >= 0.10 else "generator or evaluator"
            ),
            "EndAnswer": end_to_end["answer"],
            "OracleAnswer": oracle_answer,
        }
    )

oracle_ablation = pd.DataFrame(oracle_rows)
display(
    oracle_ablation[
        ["Query", "GoldAt1", "EndToEnd", "GoldContext", "Gain", "LikelyBottleneck"]
    ].style.format(
        {"EndToEnd": "{:.3f}", "GoldContext": "{:.3f}", "Gain": "{:+.3f}"}
    )
)

mean_gain = float(oracle_ablation["Gain"].mean())
retriever_cases = int((oracle_ablation["LikelyBottleneck"] == "retriever").sum())
print(f"Mean answer-signal gain from gold context: {mean_gain:+.3f}")
print(f"Cases pointing first to retrieval:        {retriever_cases}/{len(TEST_CASES)}")
print(
    "Read this row by row: a large gold-context gain says the answer path can use better evidence; "
    "a small gain says retrieval is not the only roadblock."
)

---

## Part 7 - From Proxies to Production: LLM-as-Judge

**Riverside's question for this section:** these proxy metrics are cheap and code-only. What do we actually turn on once real editors and real manuscripts are on the line?

### Why each proxy eventually breaks

Each proxy we built approximates a harder problem:

| Metric | Proxy | True problem | Why the proxy breaks |
| --- | --- | --- | --- |
| Retrieval relevance | Embedding cosine | Semantic entailment: does the retrieved document answer the question? | Topically adjacent but non-answering documents can look similar |
| Groundedness | Token recall | Does the context entail each answer claim? | Synonyms and negation can reuse context words in the wrong sense |
| Answer relevance | Question-answer cosine | Does the answer address the user's intent? | A hallucination about the right topic can still score highly |
| ROUGE-L | Longest shared subsequence | Are the answer's assertions semantically equivalent to the reference? | Correct paraphrases with different vocabulary score low |

### The LLM-as-Judge pattern

Production evaluation replaces the scorer, not the metric structure. Each evaluator still asks the same bounded question; the question is now posed to a strong LLM with a structured output schema:

```text
Groundedness judge:
  inputs:  FACTS = retrieved context, ANSWER = system response
  output:  { grounded: bool, rationale: str }
  question: "Are all claims in ANSWER entailed by FACTS?"

Answer relevance judge:
  inputs:  QUESTION = user query, ANSWER = system response
  output:  { relevant: bool, rationale: str }
  question: "Does ANSWER address QUESTION?"
```

The judge can detect semantic entailment, coreference, and pragmatic implication that keyword and embedding proxies miss. The rationale is deliberately bounded to one or two rubric-grounded sentences that cite the decisive evidence. Riverside needs an auditable verdict, not a request for hidden chain-of-thought.

### Mapping this notebook to LangSmith evaluators

The playground companion (`playground/af-advanced-ai/D2-rag_evaluation.ipynb`) implements this pattern end to end using LangSmith. The mapping is one-to-one:

| Function in this notebook | LangSmith evaluator | Key comparison |
| --- | --- | --- |
| `retrieval_relevance(q, docs)` | `retrieval_relevance` | query vs. `outputs["documents"]` |
| `groundedness(a, ctx)` | `groundedness` | `outputs["answer"]` vs. `outputs["documents"]` |
| `rouge_l(a, ref)` | `correctness` | `outputs["answer"]` vs. `reference_outputs["answer"]` |
| `answer_relevance(q, a)` | `relevance` | `inputs["question"]` vs. `outputs["answer"]` |

In [ ]:
#  Production evaluation pattern (LangSmith) - read-through
#
# This cell shows the LangSmith groundedness judge pattern from
# playground/af-advanced-ai/D2-rag_evaluation.ipynb.
# Requires LANGSMITH_API_KEY and OPENAI_API_KEY.

import os

LANGSMITH_KEY = os.environ.get("LANGSMITH_API_KEY")
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")

_PATTERN = [
    "# from langsmith import Client",
    "# from langchain_openai import ChatOpenAI",
    "# from typing_extensions import Annotated, TypedDict",
    "#",
    "# class GroundednessGrade(TypedDict):",
    "#     rationale: Annotated[str, ..., 'One or two rubric-grounded sentences']",
    "#     grounded:  Annotated[bool, ..., 'True if every answer claim is entailed by context']",
    "#",
    "# grader = ChatOpenAI(model='gpt-4o-mini', temperature=0)",
    "#          .with_structured_output(GroundednessGrade)",
    "#",
    "# def groundedness_judge(inputs, outputs):",
    "#     ctx = chr(10).join(d.page_content for d in outputs['documents'])",
    "#     answer = outputs['answer']",
    "#     grade = grader.invoke([",
    "#         {'role': 'system', 'content': 'Grade entailment. Return a verdict and a concise rubric rationale citing the decisive facts; do not provide hidden reasoning.'},",
    "#         {'role': 'user', 'content': 'FACTS:\\n' + ctx + '\\n\\nANSWER:\\n' + answer},",
    "#     ])",
    "#     return {'key': 'groundedness', 'score': grade['grounded'], 'comment': grade['rationale']}",
    "#",
    "# experiment = client.evaluate(",
    "#     rag_bot, data='RAG Test Evaluation',",
    "#     evaluators=[correctness_judge, groundedness_judge, relevance_judge],",
    "# )",
]

if LANGSMITH_KEY and OPENAI_KEY:
    print("API keys found. Remove the guards above to run live LangSmith evaluation.")
else:
    print("No API keys set. Production pattern shown as a code reference:\n")
    print("\n".join(_PATTERN))

# Summary: proxy means vs what an LLM judge would give
print(f"\nProxy metric means on the standard RAG bot ({len(TEST_CASES)} queries):")
for name, vals in [
    ("Retrieval Relevance", rr_scores),
    (
        "Groundedness",
        [
            groundedness(
                rag_bot(tc["question"])["answer"],
                " ".join(rag_bot(tc["question"])["retrieved_docs"]),
            )
            for tc in TEST_CASES
        ],
    ),
    ("Answer Relevance", ar_bot),
    ("ROUGE-L", rl_bot),
]:
    print(f"  {name:<22}  {np.mean(vals):.3f}")

print()
print("An LLM judge scores paraphrased correct answers higher than ROUGE-L.")
print("It scores synonym hallucinations lower than token recall allows.")
print("Use proxies for development-time regression detection;")
print("use audited LLM judges for evaluation results that inform product decisions.")

### Judge Biases: Why "Just Ask an LLM" Isn't Automatically Trustworthy

**Riverside's question:** if we swap the proxies for a real LLM judge, do we just trust whatever
it says?

Swapping a word-overlap proxy for an LLM judge fixes the semantic blind spots from Parts 2-5, but
it trades them for a different, well-documented set of failure modes specific to LLM graders:

- **Verbosity bias** — many LLM judges rate longer, more elaborate answers higher even when a
  shorter answer is equally (or more) correct, because fluent elaboration reads as competence. The
  toy demonstration below reconstructs this concretely, since it's the cheapest of the three to
  illustrate without a second model or a reorderable pairwise-judging harness.

Two related failure modes — position bias (verdict shifts with answer order in pairwise judging)
and self-preference bias (a judge favors outputs that resemble its own style) — are well documented
but need a second model or a reorderable harness this notebook doesn't build, so only verbosity
bias gets a runnable illustration below.


In [ ]:
#  Toy illustration: verbosity bias in a naive length-based judge
# A "judge" that scores by elaboration length (a crude stand-in for an LLM judge
# swayed by fluent padding) versus what the groundedness/answer-relevance proxies
# already built earlier in this notebook actually measure.

concise_answer = (
    "ReAct interleaves thought and action steps, using tools like Wikipedia search."
)
verbose_answer = (
    "ReAct interleaves thought and action steps, using tools like Wikipedia search. "
    "This is a powerful and important paradigm because it allows language models to "
    "engage in sophisticated multi-step reasoning while dynamically incorporating "
    "external information, which represents a significant advance in how such systems "
    "can be designed and deployed in real-world applications."
)


def naive_length_judge(answer):

    # Crude stand-in for a verbosity-biased judge: score scales with word count.
    return min(1.0, len(answer.split()) / 40)


concise_len_score = naive_length_judge(concise_answer)
verbose_len_score = naive_length_judge(verbose_answer)

concise_ground = groundedness(concise_answer, GOLD_CTX)
verbose_ground = groundedness(verbose_answer, GOLD_CTX)
concise_relev = answer_relevance(Q_THREAD, concise_answer)
verbose_relev = answer_relevance(Q_THREAD, verbose_answer)

print(f"{'':<12} {'Length judge':>13} {'Groundedness':>13} {'Ans Relevance':>14}")
print("-" * 56)
print(
    f"{'Concise':<12} {concise_len_score:>13.3f} {concise_ground:>13.3f} {concise_relev:>14.3f}"
)
print(
    f"{'Verbose':<12} {verbose_len_score:>13.3f} {verbose_ground:>13.3f} {verbose_relev:>14.3f}"
)

print()
print(
    "The naive length-based 'judge' scores the padded answer higher purely because it's"
)
print(
    "longer -- the extra sentences add no new grounded fact and don't move the question"
)
print(
    "any further from off-topic. Groundedness and answer relevance, built earlier in this"
)
print(
    "notebook, are far closer for the two answers because neither metric rewards length"
)
print("on its own.")
print()
print(
    "  -> a real LLM judge is far more sophisticated than this toy length heuristic, but"
)
print(
    "     the well-documented verbosity bias in LLM-as-judge research is this same failure"
)
print(
    "     mode in a subtler form: elaboration reads as competence even when it adds no new"
)
print("     grounded content.")

---

### Two Release Questions the Average Cannot Answer

The four quality metrics can look healthy while Riverside still fails two product promises.

**Citation correctness:** an answer may be grounded in the retrieved context but point the reader to the wrong source. If an answer about `RIGHTS-17` cites the parental-leave policy, the prose can be factually correct and the audit trail can still be unusable.

A citation check asks three concrete questions:

1. Did the answer include a citation when one was required?
2. Does that citation identify a passage that was actually retrieved and authorized?
3. Does the cited passage support the claim attached to it?

**Refusal appropriateness:** a safe assistant must refuse requests that are unsupported, unauthorized, or harmful, but it must not hide behind refusal on ordinary policy questions. `Reveal an unpublished manuscript outside the approved system` should be refused. `How many weeks of parental leave are available?` should not.

These are not bonus points in the composite average. They are separate release gates: a wrong citation or a required refusal failure cannot be traded away by a high answer-relevance score.

This notebook names the output contract and carries generic safety observations into the production report. A real citation/refusal suite needs outputs with stable cited passage IDs plus labeled `answer`, `abstain`, and `refuse` expectations. Riverside should add those slices to the versioned regression dataset rather than infer them from the four proxy means.

---

## Summary - The Complete RAG Evaluation Journey

### Journey completed - roadmap revisited

| Step | Concept | Claim proved |
| --- | --- | --- |
| 1 | Four RAG failure modes | Fluent answers can hide retriever, generator, or combined failure |
| 2 | Retrieval quality | Embedding similarity separates meaning better than keyword overlap, but remains a proxy |
| 3 | Groundedness | Token recall catches obvious invention and exposes its own negation/synonym boundary |
| 4 | Answer relevance | Topic alignment detects off-topic output but can reward an on-topic hallucination |
| 5 | Correctness with ROUGE-L | Shared sequence order gives paraphrases more credit than exact token sets |
| 6 | Composite dashboard | A metric fingerprint points to the weak component; the mean alone does not diagnose it |
| 6b | Oracle-context ablation | Forcing the gold document distinguishes retrieval loss from answer-path loss |
| 7 | LLM-as-judge | The evaluation question stays fixed while an audited semantic scorer replaces a weak proxy |
| Release boundary | Citations and refusals | Correct attribution and appropriate refusal remain separate gates, not bonus points in an average |

### Key insights to keep

**The taxonomy is the debugging tool.** Low retrieval relevance sends Riverside to the index and embedding model. Low groundedness sends it to the prompt or generation constraints. Low answer relevance sends it to the retrieval-generation interface. Low reference correctness sends it to factual coverage.

**An oracle ablation removes one moving part.** If the same generator improves when handed the gold document, retrieval is the first repair. If it does not, better retrieval alone will not fix the answer.

**Reference-free metrics are operational signals, not free truth.** They can run on privacy-reviewed live samples without a reference answer, but thresholds still require labeled audits and drift review.

**The proxy boundary is semantic entailment.** Negation, coreference, and subtle factual distortion are where a bounded LLM judge earns its cost. Its output should be a verdict plus concise rubric rationale, not hidden chain-of-thought.

**Critical gates stay separate.** A wrong citation, required-refusal failure, or safety failure cannot be averaged away by strong relevance.

### Evaluation checklist for a new RAG system

- [ ] Versioned labeled questions, references, and gold document IDs
- [ ] End-to-end retrieval, groundedness, relevance, and correctness by query slice
- [ ] Gold-context ablation for weak cases
- [ ] Unsupported and unauthorized questions with expected abstain/refuse behavior
- [ ] Citation IDs checked against retrieved, authorized supporting passages
- [ ] LLM-judge audit for proxy-borderline cases
- [ ] One-case sensitivity or broader uncertainty analysis before release decisions
- [ ] Evaluator agreement and calibration in the dedicated LLM-evaluation track

---

**Further reading:**

- RAGAS (Es et al. 2023): "Automated Evaluation of Retrieval Augmented Generation"
- TruLens: the faithfulness, answer-relevance, and context-relevance triad
- LangSmith evaluation guide: structured-output graders and experiment tracking
- Companion notebook: `playground/af-advanced-ai/D2-rag_evaluation.ipynb`

---

## What This Notebook Covered (and What It Didn't)

Every RAG-evaluation technique relevant to Riverside's rollout is classified below so "mentioned" never implies "fully built."

### Implemented and Demonstrated

- **Context recall@1** - binary gold-document hit rate, verified on all eight labeled queries.
- **Retrieval/context relevance** - embedding-cosine proxy tested against the four failure cases and all eight live pipeline cases.
- **Groundedness/faithfulness** - token-recall proxy tested against obvious invention, synonym distortion, and a verbatim-copy gaming case.
- **Answer relevance** - question-answer cosine tested against the four failure cases and all eight pipeline cases.
- **Reference correctness** - Jaccard strawman and hand-built ROUGE-L/LCS comparison, including a paraphrase case.
- **Composite and per-query dashboard** - metric fingerprints, radar charts, and weakest-component diagnosis.
- **Multi-metric failure fingerprinting** - dual heatmaps showing that each failure mode drops a different signal.
- **Oracle-context ablation** - each query is scored end to end and with its gold document forced into the same generator, exposing retriever versus answer-path loss.
- **Golden evaluation dataset** - eight hand-labeled `(question, reference, gold_doc_idx)` records over the 14-document corpus.
- **Threshold scorecard with one-case sensitivity** - results within one case's maximum influence become `REVIEW` rather than an automatic release conclusion.
- **Versioned production report skeleton** - offline and online aggregates carry evaluator version, dataset hash, quality gates, safety observations, latency, and cost when supplied.

### Explained But Not Fully Implemented

- **LLM-as-judge** - a syntactically valid structured-output reference pattern requests a verdict plus a one- or two-sentence rubric rationale; it is not invoked without API keys.
- **LLM-judge biases** - verbosity bias is demonstrated; position and self-preference are named but need a reorderable multi-judge harness.
- **Citation correctness and refusal appropriateness** - the release contract and required slices are defined, but the extractive teaching bot does not emit production citation IDs or refusal decisions.
- **Context Recall@k and Precision@k for multiple relevant documents** - formulas are described, while the fixture has one gold document per query.
- **ROUGE-L's remaining ceiling** - synonym-heavy paraphrases expose where sequence overlap stops representing semantic correctness.
- **Evaluation cost and latency at scale** - the production pattern records them when supplied, but this local proxy run does not simulate hosted-judge billing.

### Named But Out of Scope

- **MRR** - with one gold document and the current focus on rank-1 retrieval, it adds bookkeeping without a new diagnosis here.
- **nDCG** - requires graded relevance judgments that this binary fixture does not contain.
- **BLEU and BERTScore** - omitted to keep one clear reference-metric progression from Jaccard to ROUGE-L to semantic judge.
- **Human annotation protocol and inter-annotator agreement** - rubric design, randomized review, Cohen's kappa, and judge calibration live in `../05-llm-evaluation/02-llm-as-judge-safety-and-pipeline.ipynb`.
- **Bootstrap intervals and full confidence calibration** - the one-case review zone builds the operating intuition; formal uncertainty belongs in the dedicated evaluation track.
- **Synthetic golden-dataset generation** - the eight cases are hand-written and inspectable rather than LLM-generated.
- **A/B testing competing pipeline variants** - this notebook diagnoses one hybrid pipeline; production experiments need versioned candidates and matched traffic.

If a technique appears in the prose and is not classified here, treat that as a coverage bug, not an implied implementation.

---

## The Decision: What Does Riverside Actually Turn On in Production?

Riverside should not trust an evaluation harness against confidential manuscripts until it has exposed known failures on an inspectable rehearsal corpus. The notebook now supports an operating plan, not a claim that eight cases calibrate production.

### What Riverside monitors and gates

| Signal | Where it runs | What a drop or failure means |
| --- | --- | --- |
| Retrieval relevance | Privacy-reviewed live sample plus offline regression | The index or embedding path is returning the wrong neighborhood |
| Groundedness | Privacy-reviewed live sample plus offline regression | The answer is not tracing claims back to supplied evidence |
| Answer relevance | Privacy-reviewed live sample plus offline regression | The answer drifted from the request |
| Semantic correctness | Versioned labeled release suite | The answer omitted or changed required content |
| Citation correctness | Outputs that require stable source IDs | The answer cited a missing, unauthorized, or non-supporting passage |
| Refusal appropriateness | Labeled answer/abstain/refuse slices | The system answered when it should stop, or refused a legitimate request |
| Safety, latency, and cost | Release suite and production telemetry | The system is unsafe or operationally unsuitable even if quality looks good |

No one metric owns the release. Citation, refusal, and safety are critical gates; they cannot be averaged away by a strong relevance score.

### Diagnose before tuning

When an end-to-end case fails, rerun it with gold context. A large gain points first to retrieval. Little gain points to answer construction or the evaluator. This one ablation prevents Riverside from changing the embedding model when the generator was the actual roadblock.

### Proxies versus an LLM judge

Use the local proxies for fast regression detection and inexpensive monitoring samples. Reserve the audited LLM-judge pattern for periodic reviews and cases near a decision boundary, where entailment, negation, or paraphrase can change the conclusion. The judge returns a structured verdict plus concise rubric rationale; it does not get an open-ended request for hidden reasoning.

### Read narrow margins honestly

With eight labeled cases, one binary case represents 12.5 percentage points. The scorecard below marks any metric within one maximum case swing of its threshold as `REVIEW`. That is not a confidence interval; it is the minimum honesty required before a small fixture is allowed to influence a release.

The closing scorecard reads the live `df_dash` values, applies this review zone, and sends formal bootstrap intervals, evaluator agreement, and calibration to the dedicated LLM-evaluation track.

In [ ]:
#  Closing scorecard: thresholds plus a one-case sensitivity check
THRESH = {"RetRel": 0.5, "Groundedness": 0.4, "AnsRel": 0.4, "ROUGE-L": 0.3}

# With eight cases, one score moving from 0 to 1 can move a mean by at most 1/8 = 0.125.
one_case_swing = 1 / len(TEST_CASES)
scorecard = df_dash[list(THRESH.keys())].mean().rename("mean_score").to_frame()
scorecard["threshold"] = [THRESH[metric] for metric in scorecard.index]
scorecard["margin"] = scorecard["mean_score"] - scorecard["threshold"]


def read_gate_margin(margin):
    if margin >= one_case_swing:
        return "CLEAR"
    if margin <= -one_case_swing:
        return "BELOW THRESHOLD"
    return "REVIEW"


scorecard["status"] = [read_gate_margin(margin) for margin in scorecard["margin"]]
print(f"Closing scorecard (this run, {len(TEST_CASES)} labeled queries):\n")
print(scorecard.round(3).to_string())
print(
    f"\nOne-case sensitivity band: +/-{one_case_swing:.3f}. "
    "Inside that band, one labeled case could change the gate reading."
)

review_metrics = scorecard.index[scorecard["status"] == "REVIEW"].tolist()
below_metrics = scorecard.index[scorecard["status"] == "BELOW THRESHOLD"].tolist()
if below_metrics:
    print("HOLD: clearly below threshold -> " + ", ".join(below_metrics))
if review_metrics:
    print("REVIEW: inspect cases and add evidence -> " + ", ".join(review_metrics))
if not below_metrics and not review_metrics:
    print("CLEAR on this fixture: every metric is more than one-case swing above its threshold.")

print(
    "This is not a confidence interval. It is a practical warning against making a release decision "
    "when one example can reverse the conclusion. Bootstrap intervals and evaluator calibration "
    "belong in the dedicated LLM-evaluation track."
)

print("\nWhat ships with the assistant:")
print(
    "  1. Retrieval Relevance, Groundedness, and Answer Relevance run on privacy-reviewed live samples."
)
print("  2. Correctness runs against the versioned labeled Q&A set as a release gate.")
print(
    "  3. LLM-as-judge is reserved for periodic audits and review-zone queries, not all traffic."
)

### Key insights to keep

- **A metric that games easily is worse than no metric.** Groundedness alone rewards verbatim copying; the composite dashboard exists because no single proxy is safe to gate a release on by itself.
- **Reference-free does not mean label-free truth.** Retrieval relevance, groundedness, and answer relevance can monitor privacy-reviewed live samples without reference answers, but their thresholds still need labeled audits and drift checks.
- **The proxy boundary is where the LLM judge earns its cost.** Everything up to obvious topic drift and unsupported vocabulary is caught cheaply; past negation, subtle distortion, and coreference, spend the inference tokens.
- **Eight labeled cases are a rehearsal, not a calibration.** One binary case represents 12.5 percentage points. If one case could flip the decision, inspect the cases and add evidence instead of promoting by a narrow average.
- **The corpus was a rehearsal, not the target.** Every function above follows the same evaluation contract Riverside will use on the manuscript catalog, but the real dataset, evaluators, and thresholds must be versioned and reviewed.

Riverside does not get proof that the assistant will never hallucinate; no evaluation harness gives that. It gets a diagnostic path, a review zone that resists false precision, and a monitoring plan the team can afford to operate.

---

## Production and Cloud RAG Evaluation

Production evaluation combines two loops:

- **Offline regression:** run a versioned, reviewed dataset before release. Record the dataset version and content hash, compare the candidate with the current baseline, and block promotion when quality, safety, latency, or cost gates fail.
- **Online telemetry:** sample privacy-reviewed production traces, monitor reference-free quality metrics plus safety outcomes, latency, and cost, and send difficult or drifting cases back into the labeled regression set.

Datasets, prompts, retrievers, generators, evaluator versions, thresholds, and reports should be immutable release artifacts. CI/CD promotes only candidates whose required gates pass; otherwise it keeps the previous version serving. A post-release breach should trigger alerting and rollback through the deployment platform, while the persisted report provides the evidence for the decision.

The cells below reuse this notebook's `TEST_CASES`, `rag_bot`, and four evaluators. They are cloud-neutral: production systems may add real `safety_passed` and `cost_usd` fields to each response, but no provider API is simulated here.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from time import perf_counter
from typing import Callable, Mapping, Sequence
import json


@dataclass(frozen=True)
class ProductionEvalConfig:
    dataset_version: str = "rag-regression-v1"
    evaluator_version: str = "notebook-proxies-v1"
    report_dir: Path = Path("artifacts/rag-evaluation")
    offline_quality_thresholds: Mapping[str, float] = field(
        default_factory=lambda: {
            "RetRel": 0.50,
            "Groundedness": 0.40,
            "AnsRel": 0.40,
            "ROUGE-L": 0.30,
        }
    )
    online_quality_thresholds: Mapping[str, float] = field(
        default_factory=lambda: {
            "RetRel": 0.50,
            "Groundedness": 0.40,
            "AnsRel": 0.40,
        }
    )
    min_safety_pass_rate: float = 0.99
    max_p95_latency_ms: float = 2500.0
    max_mean_cost_usd: float = 0.02
    require_safety_observations: bool = True
    require_cost_observations: bool = True


RUN_PRODUCTION_OFFLINE_EVAL = False
RUN_PRODUCTION_ONLINE_AGGREGATION = False
PRODUCTION_EVAL_CONFIG = ProductionEvalConfig()

In [ ]:
def validate_regression_dataset(dataset: Sequence[Mapping[str, object]]) -> None:
    if not dataset:
        raise ValueError("The regression dataset is empty.")
    required = {"question", "reference"}
    for index, case in enumerate(dataset):
        missing = required - case.keys()
        if missing:
            raise ValueError(f"Case {index} is missing fields: {sorted(missing)}")


def dataset_fingerprint(dataset: Sequence[Mapping[str, object]]) -> str:
    canonical = json.dumps(
        list(dataset), sort_keys=True, separators=(",", ":"), ensure_ascii=True
    )
    return sha256(canonical.encode("utf-8")).hexdigest()


def evaluate_offline_regression(
    system_fn: Callable[[str], Mapping[str, object]],
    dataset: Sequence[Mapping[str, object]],
) -> pd.DataFrame:
    """Run this notebook's evaluators against the labeled data contract."""
    validate_regression_dataset(dataset)
    rows = []

    for case in dataset:
        question = str(case["question"])
        started = perf_counter()
        output = dict(system_fn(question))
        latency_ms = (perf_counter() - started) * 1000

        answer = str(output["answer"])
        retrieved_docs = [str(doc) for doc in output["retrieved_docs"]]
        context = " ".join(retrieved_docs)
        row = {
            "Query": question,
            "Reference": str(case["reference"]),
            "Answer": answer,
            "RetRel": retrieval_relevance(question, retrieved_docs),
            "Groundedness": groundedness(answer, context),
            "AnsRel": answer_relevance(question, answer),
            "ROUGE-L": rouge_l(answer, str(case["reference"])),
            "latency_ms": latency_ms,
            "cost_usd": output.get("cost_usd"),
            "safety_passed": output.get("safety_passed"),
        }
        row["Composite"] = float(
            np.mean([row["RetRel"], row["Groundedness"], row["AnsRel"], row["ROUGE-L"]])
        )
        rows.append(row)

    return pd.DataFrame(rows)


def _numeric_observations(rows: pd.DataFrame, column: str) -> np.ndarray:
    if column not in rows:
        return np.array([], dtype=float)
    return pd.to_numeric(rows[column], errors="coerce").dropna().to_numpy(dtype=float)


def _safety_observations(rows: pd.DataFrame) -> np.ndarray:
    if "safety_passed" not in rows:
        return np.array([], dtype=float)
    normalized = rows["safety_passed"].map(
        lambda value: value
        if isinstance(value, bool)
        else {"true": True, "false": False, "1": True, "0": False}.get(
            str(value).strip().lower()
        )
    )
    return normalized.dropna().astype(float).to_numpy()


def build_gate_report(
    rows: pd.DataFrame,
    config: ProductionEvalConfig,
    quality_thresholds: Mapping[str, float],
    run_type: str,
    data_sha256: str,
) -> dict:
    if rows.empty:
        raise ValueError("Cannot build a gate report from zero rows.")

    gates = []
    aggregates = {}
    for metric, threshold in quality_thresholds.items():
        values = _numeric_observations(rows, metric)
        observed = float(values.mean()) if values.size else None
        aggregates[f"mean_{metric}"] = observed
        gates.append(
            {
                "name": f"quality:{metric}",
                "observed": observed,
                "threshold": threshold,
                "passed": observed is not None and observed >= threshold,
            }
        )

    safety_values = _safety_observations(rows)
    safety_rate = float(safety_values.mean()) if safety_values.size else None
    aggregates["safety_pass_rate"] = safety_rate
    gates.append(
        {
            "name": "safety_pass_rate",
            "observed": safety_rate,
            "threshold": config.min_safety_pass_rate,
            "passed": (
                safety_rate >= config.min_safety_pass_rate
                if safety_rate is not None
                else not config.require_safety_observations
            ),
        }
    )

    latency_values = _numeric_observations(rows, "latency_ms")
    p95_latency = float(np.percentile(latency_values, 95)) if latency_values.size else None
    aggregates["p95_latency_ms"] = p95_latency
    gates.append(
        {
            "name": "p95_latency_ms",
            "observed": p95_latency,
            "threshold": config.max_p95_latency_ms,
            "passed": p95_latency is not None and p95_latency <= config.max_p95_latency_ms,
        }
    )

    cost_values = _numeric_observations(rows, "cost_usd")
    mean_cost = float(cost_values.mean()) if cost_values.size else None
    aggregates["mean_cost_usd"] = mean_cost
    gates.append(
        {
            "name": "mean_cost_usd",
            "observed": mean_cost,
            "threshold": config.max_mean_cost_usd,
            "passed": (
                mean_cost <= config.max_mean_cost_usd
                if mean_cost is not None
                else not config.require_cost_observations
            ),
        }
    )

    passed = all(gate["passed"] for gate in gates)
    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "run_type": run_type,
        "dataset_version": config.dataset_version,
        "dataset_sha256": data_sha256,
        "evaluator_version": config.evaluator_version,
        "row_count": len(rows),
        "aggregates": aggregates,
        "gates": gates,
        "decision": "PROMOTE" if passed else "HOLD_OR_ROLL_BACK",
    }


def persist_evaluation_report(
    rows: pd.DataFrame, report: Mapping[str, object], report_dir: Path
) -> Mapping[str, Path]:
    report_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    stem = f"{report['run_type']}-{report['dataset_version']}-{stamp}"
    csv_path = report_dir / f"{stem}.csv"
    json_path = report_dir / f"{stem}.json"
    csv_temp = csv_path.with_suffix(".csv.tmp")
    json_temp = json_path.with_suffix(".json.tmp")

    rows.to_csv(csv_temp, index=False)
    json_temp.write_text(
        json.dumps(dict(report), indent=2, sort_keys=True), encoding="utf-8"
    )
    csv_temp.replace(csv_path)
    json_temp.replace(json_path)
    return {"rows": csv_path, "report": json_path}

In [ ]:
if RUN_PRODUCTION_OFFLINE_EVAL:
    offline_rows = evaluate_offline_regression(rag_bot, TEST_CASES)
    offline_report = build_gate_report(
        rows=offline_rows,
        config=PRODUCTION_EVAL_CONFIG,
        quality_thresholds=PRODUCTION_EVAL_CONFIG.offline_quality_thresholds,
        run_type="offline-regression",
        data_sha256=dataset_fingerprint(TEST_CASES),
    )
    offline_artifacts = persist_evaluation_report(
        offline_rows, offline_report, PRODUCTION_EVAL_CONFIG.report_dir
    )
    print(json.dumps(offline_report, indent=2))
    print(f"Persisted: {offline_artifacts}")
else:
    print("Offline production evaluation disabled: RUN_PRODUCTION_OFFLINE_EVAL = False")


if RUN_PRODUCTION_ONLINE_AGGREGATION:
    telemetry_path = PRODUCTION_EVAL_CONFIG.report_dir / "production-telemetry.csv"
    if not telemetry_path.exists():
        raise FileNotFoundError(
            f"Expected privacy-reviewed telemetry at {telemetry_path}."
        )

    online_rows = pd.read_csv(telemetry_path)
    online_report = build_gate_report(
        rows=online_rows,
        config=PRODUCTION_EVAL_CONFIG,
        quality_thresholds=PRODUCTION_EVAL_CONFIG.online_quality_thresholds,
        run_type="online-telemetry",
        data_sha256=sha256(telemetry_path.read_bytes()).hexdigest(),
    )
    online_artifacts = persist_evaluation_report(
        online_rows, online_report, PRODUCTION_EVAL_CONFIG.report_dir
    )
    print(json.dumps(online_report, indent=2))
    print(f"Persisted: {online_artifacts}")
else:
    print(
        "Online telemetry aggregation disabled: "
        "RUN_PRODUCTION_ONLINE_AGGREGATION = False"
    )